# Titan Experiment Runner
This notebook sets up the environment and runs the Associative Recall experiment on the Titans model.

## 1. Install Dependencies

In [ ]:
# Install dependencies
!pip install torch deepspeed transformers tqdm einops sentence-transformers

## 2. Setup File System

In [ ]:
import os
import base64

print(f'Current working directory: {os.getcwd()}')

def write_file(path, b64_content):
    target_path = os.path.abspath(path)
    directory = os.path.dirname(target_path)
    if directory:
        os.makedirs(directory, exist_ok=True)
    with open(target_path, "wb") as f:
        f.write(base64.b64decode(b64_content))
    print(f"Created {target_path}")

files = {
    'src/main.py': '...',
    'projects/original_benchmarks/associative_recall.py': 'DQppbXBvcnQgdG9yY2gNCmZyb20gdG9yY2gudXRpbHMuZGF0YSBpbXBvcnQgRGF0YXNldA0KaW1wb3J0IHJhbmRvbQ0KDQpjbGFzcyBBc3NvY2lhdGl2ZVJlY2FsbERhdGFzZXQoRGF0YXNldCk6DQogICAgZGVmIF9faW5pdF9fKHNlbGYsIHZvY2FiX3NpemU9MTAwLCBzZXFfbGVuPTEyOCwgbnVtX3NhbXBsZXM9MTAwMDAsIHBhaXJfbGVuPTIpOg0KICAgICAgICAiIiINCiAgICAgICAgQXJnczoNCiAgICAgICAgICAgIHZvY2FiX3NpemU6IFNpemUgb2YgdGhlIHZvY2FidWxhcnkgKGtleXMgYW5kIHZhbHVlcykuDQogICAgICAgICAgICBzZXFfbGVuOiBUb3RhbCBsZW5ndGggb2YgdGhlIHNlcXVlbmNlLg0KICAgICAgICAgICAgbnVtX3NhbXBsZXM6IE51bWJlciBvZiBzYW1wbGVzIGluIHRoZSBkYXRhc2V0Lg0KICAgICAgICAgICAgcGFpcl9sZW46IE51bWJlciBvZiB0b2tlbnMgcGVyIGtleS12YWx1ZSBwYWlyICh1c3VhbGx5IDI6IGtleSwgdmFsdWUpLg0KICAgICAgICAiIiINCiAgICAgICAgc2VsZi52b2NhYl9zaXplID0gdm9jYWJfc2l6ZQ0KICAgICAgICBzZWxmLnNlcV9sZW4gPSBzZXFfbGVuDQogICAgICAgIHNlbGYubnVtX3NhbXBsZXMgPSBudW1fc2FtcGxlcw0KICAgICAgICBzZWxmLnBhaXJfbGVuID0gcGFpcl9sZW4NCiAgICAgICAgDQogICAgICAgICMgUmVzZXJ2ZSBleHBsaWNpdCB0b2tlbnMNCiAgICAgICAgc2VsZi5RVUVSWV9UT0tFTiA9IHZvY2FiX3NpemUgLSAxICAjIE5vdCBzdHJpY3RseSBuZWVkZWQgaWYgcG9zaXRpb24gaW1wbGljaXRseSBkZWZpbmVzIHF1ZXJ5LCBidXQgZ29vZCBmb3IgY2xhcml0eSBpZiB3ZSB1c2VkIHNwZWNpYWwgdG9rZW5zDQogICAgICAgICMgRm9yIHNpbXBsaWNpdHkgaW4gdGhpcyBudW1lcmljIHRhc2ssIHdlIGp1c3QgdXNlIHRoZSBudW1iZXJzIFswLCB2b2NhYl9zaXplLTJdIGZvciBkYXRhDQogICAgICAgIA0KICAgIGRlZiBfX2xlbl9fKHNlbGYpOg0KICAgICAgICByZXR1cm4gc2VsZi5udW1fc2FtcGxlcw0KDQogICAgZGVmIF9fZ2V0aXRlbV9fKHNlbGYsIGlkeCk6DQogICAgICAgICMgMS4gR2VuZXJhdGUgdW5pcXVlIGtleXMNCiAgICAgICAgIyBXZSBuZWVkIGVub3VnaCBrZXlzIHRvIGZpbGwgbW9zdCBvZiB0aGUgc2VxdWVuY2UuDQogICAgICAgICMgTWF4IHBhaXJzIGFwcHJveCBzZXFfbGVuIC8gMg0KICAgICAgICBudW1fcGFpcnMgPSAoc2VsZi5zZXFfbGVuIC0gMSkgLy8gMiANCiAgICAgICAgDQogICAgICAgICMgRW5zdXJlIHZvY2FiIHNpemUgaXMgc3VmZmljaWVudCBmb3IgdW5pcXVlIGtleXMgaWYgcG9zc2libGUsIG9yIGFsbG93IHJlcGVhdHMgaWYgdm9jYWIgaXMgc21hbGwgKGJ1dCBBUiB1c3VhbGx5IHJlcXVpcmVzIHVuaXF1ZSBrZXlzIGZvciBxdWVyeWluZykNCiAgICAgICAgIyBMZXQncyBhc3N1bWUgdm9jYWJfc2l6ZSA+PiBudW1fcGFpcnMgdXN1YWxseSwgb3Igd2Ugc2FtcGxlIHdpdGggcmVwbGFjZW1lbnQgYnV0IHRoYXQgbWFrZXMgdGFzayBoYXJkZXIvYW1iaWd1b3VzLg0KICAgICAgICAjIFN0YW5kYXJkIEFSOiBrZXlzIGFyZSB1bmlxdWUuDQogICAgICAgIA0KICAgICAgICBhdmFpbGFibGVfa2V5cyA9IGxpc3QocmFuZ2Uoc2VsZi52b2NhYl9zaXplIC0gMTApKSAjIFJlc2VydmUgc29tZSB0b3AgdG9rZW5zIGZvciBzcGVjaWFsIHVzZSBpZiBuZWVkZWQNCiAgICAgICAgaWYgbGVuKGF2YWlsYWJsZV9rZXlzKSA8IG51bV9wYWlyczoNCiAgICAgICAgICAgICMgSWYgdm9jYWIgaXMgc21hbGwsIHdlIHJlZHVjZSBudW1fcGFpcnMNCiAgICAgICAgICAgIG51bV9wYWlycyA9IGxlbihhdmFpbGFibGVfa2V5cykNCiAgICAgICAgICAgIA0KICAgICAgICBrZXlzID0gcmFuZG9tLnNhbXBsZShhdmFpbGFibGVfa2V5cywgbnVtX3BhaXJzKQ0KICAgICAgICB2YWx1ZXMgPSBbcmFuZG9tLmNob2ljZShhdmFpbGFibGVfa2V5cykgZm9yIF8gaW4gcmFuZ2UobnVtX3BhaXJzKV0NCiAgICAgICAgDQogICAgICAgICMgMi4gU2VsZWN0IGEgdGFyZ2V0IHF1ZXJ5DQogICAgICAgIHRhcmdldF9pZHggPSByYW5kb20ucmFuZGludCgwLCBudW1fcGFpcnMgLSAxKQ0KICAgICAgICBxdWVyeV9rZXkgPSBrZXlzW3RhcmdldF9pZHhdDQogICAgICAgIHRhcmdldF92YWwgPSB2YWx1ZXNbdGFyZ2V0X2lkeF0NCiAgICAgICAgDQogICAgICAgICMgMy4gQ29uc3RydWN0IHNlcXVlbmNlDQogICAgICAgICMgaW5wdXQ6IGsxIHYxIGsyIHYyIC4uLiBrTiB2TiBxdWVyeV9rZXkNCiAgICAgICAgIyBsYWJlbDogLTEgLTEgLTEgLTEgLi4uIC0xIC0xIHRhcmdldF92YWwNCiAgICAgICAgDQogICAgICAgIGlucHV0X2lkcyA9IFtdDQogICAgICAgIGxhYmVscyA9IFtdDQogICAgICAgIA0KICAgICAgICBmb3IgaywgdiBpbiB6aXAoa2V5cywgdmFsdWVzKToNCiAgICAgICAgICAgIGlucHV0X2lkcy5leHRlbmQoW2ssIHZdKQ0KICAgICAgICAgICAgbGFiZWxzLmV4dGVuZChbLTEwMCwgLTEwMF0pICMgSWdub3JlIGxvc3MgZm9yIGNvbnRleHQNCiAgICAgICAgICAgIA0KICAgICAgICAjIEFwcGVuZCBxdWVyeQ0KICAgICAgICBpbnB1dF9pZHMuYXBwZW5kKHF1ZXJ5X2tleSkNCiAgICAgICAgbGFiZWxzLmFwcGVuZCh0YXJnZXRfdmFsKQ0KICAgICAgICANCiAgICAgICAgIyBQYWRkaW5nIGlmIG5lZWRlZCAoc2ltcGxpZmllZDoganVzdCB0cnVuY2F0ZSBvciBhc3N1bWUgZml4ZWQgc2l6ZSBsb2dpYyBtYXRjaGVzKQ0KICAgICAgICAjIEZvciB0aGlzIGRlbW8sIHdlIHJldHVybiB0ZW5zb3JzIGRpcmVjdGx5Lg0KICAgICAgICAjIElmIHRoZSBnZW5lcmF0ZWQgbGVuZ3RoIDwgc2VsZi5zZXFfbGVuLCB3ZSBtaWdodCBwYWQgKGJ1dCBiYXRjaGluZyBoYW5kbGVzIGl0IGlmIHdlIHVzZSBjb2xsYXRlKS4NCiAgICAgICAgIyBMZXQncyBzaW1wbGUgcmV0dXJuIGFzIGlzLCBhbmQgdGhlIG1vZGVsIGhhbmRsZXMgdmFyaWFibGUgbGVuZ3RoIG9yIHdlIG1ha2UgaXQgZml4ZWQuDQogICAgICAgIA0KICAgICAgICByZXR1cm4gdG9yY2gudGVuc29yKGlucHV0X2lkcywgZHR5cGU9dG9yY2gubG9uZyksIHRvcmNoLnRlbnNvcihsYWJlbHMsIGR0eXBlPXRvcmNoLmxvbmcpDQoNCmlmIF9fbmFtZV9fID09ICJfX21haW5fXyI6DQogICAgZHMgPSBBc3NvY2lhdGl2ZVJlY2FsbERhdGFzZXQoKQ0KICAgIHgsIHkgPSBkc1swXQ0KICAgIHByaW50KCJJbnB1dDoiLCB4KQ0KICAgIHByaW50KCJMYWJlbDoiLCB5KQ0K',
    'projects/original_benchmarks/train_toy.py': 'DQppbXBvcnQgc3lzDQppbXBvcnQgb3MNCmltcG9ydCB0b3JjaA0KaW1wb3J0IHRvcmNoLm5uIGFzIG5uDQppbXBvcnQgYXJncGFyc2UNCmZyb20gdG9yY2gudXRpbHMuZGF0YSBpbXBvcnQgRGF0YUxvYWRlcg0KZnJvbSB0cWRtIGltcG9ydCB0cWRtDQoNCiMgQWRkIHBhcmVudCBkaXJlY3RvcnkgdG8gcGF0aCB0byBpbXBvcnQgbWFpbiAob3JpZ2luYWxfYmVuY2htYXJrcyAtPiBwcm9qZWN0cyAtPiBUaXRhbkxMTSkNCnN5cy5wYXRoLmFwcGVuZChvcy5wYXRoLmRpcm5hbWUob3MucGF0aC5kaXJuYW1lKG9zLnBhdGguZGlybmFtZShvcy5wYXRoLmFic3BhdGgoX19maWxlX18pKSkpKQ0Kc3lzLnBhdGguYXBwZW5kKG9zLnBhdGguam9pbihvcy5wYXRoLmRpcm5hbWUob3MucGF0aC5kaXJuYW1lKG9zLnBhdGguZGlybmFtZShvcy5wYXRoLmFic3BhdGgoX19maWxlX18pKSkpLCAnc3JjJykpDQpmcm9tIG1haW4gaW1wb3J0IFRpdGFuTUFDLCBUaXRhbk1BRywgVGl0YW5NQUwsIFRpdGFuTW9kZWxGb3JMTQ0KDQpmcm9tIGFzc29jaWF0aXZlX3JlY2FsbCBpbXBvcnQgQXNzb2NpYXRpdmVSZWNhbGxEYXRhc2V0DQoNCmRlZiB0cmFpbihhcmdzKToNCiAgICBkZXZpY2UgPSB0b3JjaC5kZXZpY2UoImN1ZGEiIGlmIHRvcmNoLmN1ZGEuaXNfYXZhaWxhYmxlKCkgZWxzZSAiY3B1IikNCiAgICBwcmludChmIlVzaW5nIGRldmljZToge2RldmljZX0iKQ0KDQogICAgIyBEYXRhc2V0DQogICAgdHJhaW5fZHMgPSBBc3NvY2lhdGl2ZVJlY2FsbERhdGFzZXQodm9jYWJfc2l6ZT1hcmdzLnZvY2FiX3NpemUsIHNlcV9sZW49YXJncy5zZXFfbGVuLCBudW1fc2FtcGxlcz0xMDAwMCkNCiAgICB0cmFpbl9kbCA9IERhdGFMb2FkZXIodHJhaW5fZHMsIGJhdGNoX3NpemU9YXJncy5iYXRjaF9zaXplLCBzaHVmZmxlPVRydWUpDQogICAgDQogICAgIyBNb2RlbCBTZXR1cA0KICAgIGlmIGFyZ3MubW9kZWxfdmFyaWFudCA9PSAiTUFDIjoNCiAgICAgICAgdGl0YW5fbW9kdWxlID0gVGl0YW5NQUMoYXJncy5kaW0sIGFyZ3MuY2h1bmtfc2l6ZSwgYXJncy5oaWRkZW5fZGltLCBhcmdzLm1lbW9yeV9kZXB0aCwgYXJncy5udW1fcGVyc2lzdGVudF90b2tlbnMsIHRocmVzaG9sZD1hcmdzLnRocmVzaG9sZCkNCiAgICBlbGlmIGFyZ3MubW9kZWxfdmFyaWFudCA9PSAiTUFHIjoNCiAgICAgICAgdGl0YW5fbW9kdWxlID0gVGl0YW5NQUcoYXJncy5kaW0sIGFyZ3MuaGlkZGVuX2RpbSwgYXJncy5tZW1vcnlfZGVwdGgsIGFyZ3MubnVtX3BlcnNpc3RlbnRfdG9rZW5zLCBhcmdzLndpbmRvd19zaXplLCB0aHJlc2hvbGQ9YXJncy50aHJlc2hvbGQpDQogICAgZWxzZTogIyBNQUwNCiAgICAgICAgdGl0YW5fbW9kdWxlID0gVGl0YW5NQUwoYXJncy5kaW0sIGFyZ3MuaGlkZGVuX2RpbSwgYXJncy5tZW1vcnlfZGVwdGgsIGFyZ3MubnVtX3BlcnNpc3RlbnRfdG9rZW5zLCBhcmdzLndpbmRvd19zaXplLCB0aHJlc2hvbGQ9YXJncy50aHJlc2hvbGQpDQogICAgICAgIA0KICAgIG1vZGVsID0gVGl0YW5Nb2RlbEZvckxNKHRpdGFuX21vZHVsZSwgYXJncy52b2NhYl9zaXplLCBhcmdzLmRpbSkNCiAgICBtb2RlbC50byhkZXZpY2UpDQogICAgDQogICAgb3B0aW1pemVyID0gdG9yY2gub3B0aW0uQWRhbVcobW9kZWwucGFyYW1ldGVycygpLCBscj0xZS00KQ0KICAgIA0KICAgIG1vZGVsLnRyYWluKCkNCiAgICANCiAgICBmb3IgZXBvY2ggaW4gcmFuZ2UoYXJncy5udW1fZXBvY2hzKToNCiAgICAgICAgdG90YWxfbG9zcyA9IDANCiAgICAgICAgY29ycmVjdCA9IDANCiAgICAgICAgdG90YWwgPSAwDQogICAgICAgIA0KICAgICAgICBwYmFyID0gdHFkbSh0cmFpbl9kbCwgZGVzYz1mIkVwb2NoIHtlcG9jaCsxfSIpDQogICAgICAgIGZvciBpbnB1dF9pZHMsIGxhYmVscyBpbiBwYmFyOg0KICAgICAgICAgICAgaW5wdXRfaWRzLCBsYWJlbHMgPSBpbnB1dF9pZHMudG8oZGV2aWNlKSwgbGFiZWxzLnRvKGRldmljZSkNCiAgICAgICAgICAgIA0KICAgICAgICAgICAgb3B0aW1pemVyLnplcm9fZ3JhZCgpDQogICAgICAgICAgICBsb2dpdHMsIGxvc3MgPSBtb2RlbChpbnB1dF9pZHMsIGxhYmVscykgIyBUaXRhbk1vZGVsRm9yTE0gaGFuZGxlcyBsb3NzIGNhbGN1bGF0aW9uIGlmIGxhYmVscyBwcm92aWRlZA0KICAgICAgICAgICAgDQogICAgICAgICAgICBsb3NzLmJhY2t3YXJkKCkNCiAgICAgICAgICAgIG9wdGltaXplci5zdGVwKCkNCiAgICAgICAgICAgIA0KICAgICAgICAgICAgdG90YWxfbG9zcyArPSBsb3NzLml0ZW0oKQ0KICAgICAgICAgICAgDQogICAgICAgICAgICAjIEFjY3VyYWN5IE1ldHJpYw0KICAgICAgICAgICAgIyBsYWJlbHMgaGF2ZSAtMTAwIGZvciBpZ25vcmVkIHBvc2l0aW9ucy4gV2Ugb25seSBjYXJlIGFib3V0IHRoZSBsYXN0IHRva2VuIHVzdWFsbHkuDQogICAgICAgICAgICAjIEZsYXR0ZW4gZm9yIGFyZ21heA0KICAgICAgICAgICAgcHJlZF90b2tlbnMgPSB0b3JjaC5hcmdtYXgobG9naXRzLCBkaW09LTEpDQogICAgICAgICAgICBtYXNrID0gbGFiZWxzICE9IC0xMDANCiAgICAgICAgICAgIA0KICAgICAgICAgICAgY29ycmVjdCArPSAocHJlZF90b2tlbnNbbWFza10gPT0gbGFiZWxzW21hc2tdKS5zdW0oKS5pdGVtKCkNCiAgICAgICAgICAgIHRvdGFsICs9IG1hc2suc3VtKCkuaXRlbSgpDQogICAgICAgICAgICANCiAgICAgICAgICAgIHBiYXIuc2V0X3Bvc3RmaXgoeyJMb3NzIjogZiJ7bG9zcy5pdGVtKCk6LjRmfSIsICJBY2MiOiBmIntjb3JyZWN0L3RvdGFsOi4yJX0ifSkNCiAgICAgICAgICAgIA0KICAgIHByaW50KCJUcmFpbmluZyBmaW5pc2hlZC4iKQ0KDQppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOg0KICAgIHBhcnNlciA9IGFyZ3BhcnNlLkFyZ3VtZW50UGFyc2VyKCkNCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLW1vZGVsX3ZhcmlhbnQiLCB0eXBlPXN0ciwgZGVmYXVsdD0iTUFDIiwgY2hvaWNlcz1bIk1BQyIsICJNQUciLCAiTUFMIl0pDQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS12b2NhYl9zaXplIiwgdHlwZT1pbnQsIGRlZmF1bHQ9MTI4KQ0KICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tc2VxX2xlbiIsIHR5cGU9aW50LCBkZWZhdWx0PTY0KQ0KICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tZGltIiwgdHlwZT1pbnQsIGRlZmF1bHQ9MTI4KQ0KICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0taGlkZGVuX2RpbSIsIHR5cGU9aW50LCBkZWZhdWx0PTEyOCkNCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLW1lbW9yeV9kZXB0aCIsIHR5cGU9aW50LCBkZWZhdWx0PTIpDQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1udW1fcGVyc2lzdGVudF90b2tlbnMiLCB0eXBlPWludCwgZGVmYXVsdD0yKQ0KICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tY2h1bmtfc2l6ZSIsIHR5cGU9aW50LCBkZWZhdWx0PTMyKQ0KICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0td2luZG93X3NpemUiLCB0eXBlPWludCwgZGVmYXVsdD0zMikNCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLWJhdGNoX3NpemUiLCB0eXBlPWludCwgZGVmYXVsdD0xNikNCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLW51bV9lcG9jaHMiLCB0eXBlPWludCwgZGVmYXVsdD01KQ0KICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tdGhyZXNob2xkIiwgdHlwZT1mbG9hdCwgZGVmYXVsdD0wLjApDQogICAgDQogICAgYXJncyA9IHBhcnNlci5wYXJzZV9hcmdzKCkNCiAgICB0cmFpbihhcmdzKQ0K',
    'projects/hybrid_titans/rag_compare.py': 'aW1wb3J0IHRvcmNoDQppbXBvcnQgYXJncGFyc2UNCmltcG9ydCByYW5kb20NCmltcG9ydCBzeXMNCmltcG9ydCBvcw0KDQojIEFkZCBwYXJlbnQgZGlyZWN0b3J5IHRvIHBhdGggdG8gaW1wb3J0IG1haW4gKGh5YnJpZF90aXRhbnMgLT4gcHJvamVjdHMgLT4gVGl0YW5MTE0pDQpzeXMucGF0aC5hcHBlbmQob3MucGF0aC5kaXJuYW1lKG9zLnBhdGguZGlybmFtZShvcy5wYXRoLmRpcm5hbWUob3MucGF0aC5hYnNwYXRoKF9fZmlsZV9fKSkpKSkNCnN5cy5wYXRoLmFwcGVuZChvcy5wYXRoLmpvaW4ob3MucGF0aC5kaXJuYW1lKG9zLnBhdGguZGlybmFtZShvcy5wYXRoLmRpcm5hbWUob3MucGF0aC5hYnNwYXRoKF9fZmlsZV9fKSkpKSwgJ3NyYycpKQ0KDQpmcm9tIG1haW4gaW1wb3J0IFRpdGFuTUFDLCBUaXRhbk1BRywgVGl0YW5NQUwsIFRpdGFuUkFHDQoNCmRlZiBnZW5lcmF0ZV9oYXlzdGFjayhudW1fZG9jcywgZG9jX2xlbiwgZGltLCBrZXlfaWR4PU5vbmUpOg0KICAgICIiIg0KICAgIEdlbmVyYXRlcyBOIGRvY3VtZW50cy4gT25lIGNvbnRhaW5pbmcgdGhlICduZWVkbGUnIChwYXNza2V5KSwgb3RoZXJzIG5vaXNlLg0KICAgICIiIg0KICAgIGRvY3MgPSBbXQ0KICAgIG5lZWRsZV9wYXR0ZXJuID0gdG9yY2gucmFuZG4oMSwgZGltKQ0KICAgIA0KICAgIGlmIGtleV9pZHggaXMgTm9uZToNCiAgICAgICAga2V5X2lkeCA9IHJhbmRvbS5yYW5kaW50KDAsIG51bV9kb2NzIC0gMSkNCiAgICAgICAgDQogICAgZm9yIGkgaW4gcmFuZ2UobnVtX2RvY3MpOg0KICAgICAgICBkb2MgPSB0b3JjaC5yYW5kbihkb2NfbGVuLCBkaW0pDQogICAgICAgIGlmIGkgPT0ga2V5X2lkeDoNCiAgICAgICAgICAgIGRvY1stMSwgOl0gPSBuZWVkbGVfcGF0dGVybg0KICAgICAgICBkb2NzLmFwcGVuZChkb2MpDQogICAgICAgIA0KICAgIHJldHVybiBkb2NzLCBuZWVkbGVfcGF0dGVybiwga2V5X2lkeA0KDQpkZWYgcHJldHJhaW5faWRlbnRpdHkobW9kZWwsIGRpbSwgc3RlcHM9NTAwKToNCiAgICAiIiINCiAgICBUcmFpbnMgdGhlIG1vZGVsJ3MgQXR0ZW50aW9uL1Byb2Nlc3NpbmcgbGF5ZXJzIHRvIHBlcmZvcm0gSWRlbnRpdHkgTWFwcGluZyBmKHgpPXguDQogICAgIiIiDQogICAgb3B0aW1pemVyID0gdG9yY2gub3B0aW0uQWRhbVcobW9kZWwucGFyYW1ldGVycygpLCBscj0wLjAxKQ0KICAgIA0KICAgIHByaW50KGYiICBbUHJlLXRyYWluaW5nXSBUZWFjaGluZyBtb2RlbCB0byBjb3B5IChJZGVudGl0eSBGdW5jdGlvbikuLi4iKQ0KICAgIA0KICAgICMgW0ZpeF0gRGlzYWJsZSBUaXRhbidzIGludGVybmFsIFRlc3QtVGltZSBUcmFpbmluZyBkdXJpbmcgSWRlbnRpdHkgUHJlLXRyYWluaW5nLg0KICAgICMgV2Ugb25seSB3YW50IHRvIHRyYWluIHRoZSBBdHRlbnRpb24gbWVjaGFuaXNtIChXZWlnaHRzKSwgbm90IHRoZSBNZW1vcnkgKFN0YXRlKS4NCiAgICAjIE90aGVyd2lzZSwgaW4tcGxhY2UgdXBkYXRlcyBjYXVzZSAnYmFja3dhcmQgdGhyb3VnaCBncmFwaCBzZWNvbmQgdGltZScgZXJyb3IuDQogICAgc2F2ZWRfdGhyZXNob2xkID0gbW9kZWwudGhyZXNob2xkDQogICAgbW9kZWwudGhyZXNob2xkID0gZmxvYXQoJ2luZicpICMgRm9yY2Ugc2tpcCB1cGRhdGUNCiAgICANCiAgICBmb3IgaSBpbiByYW5nZShzdGVwcyk6DQogICAgICAgICMgR2VuZXJhdGUgcmFuZG9tIGRhdGENCiAgICAgICAgeCA9IHRvcmNoLnJhbmRuKDEsIDMyLCBkaW0pICMgW0JhdGNoLCBTZXEsIERpbV0NCiAgICAgICAgdGFyZ2V0ID0geA0KICAgICAgICANCiAgICAgICAgb3B0aW1pemVyLnplcm9fZ3JhZCgpDQogICAgICAgIA0KICAgICAgICBwcmVkID0gbW9kZWwoeCkNCiAgICAgICAgbG9zcyA9ICgocHJlZCAtIHRhcmdldCkqKjIpLm1lYW4oKQ0KICAgICAgICBsb3NzLmJhY2t3YXJkKCkNCiAgICAgICAgb3B0aW1pemVyLnN0ZXAoKQ0KICAgICAgICANCiAgICAgICAgaWYgaSAlIDIwID09IDA6DQogICAgICAgICAgICBwcmludChmIiAgICBTdGVwIHtpfTogTG9zcyB7bG9zcy5pdGVtKCk6LjRmfSIpDQogICAgICAgICAgICANCiAgICAjIFJlc3RvcmUgb3JpZ2luYWwgdGhyZXNob2xkIGZvciBFeHBlcmltZW50DQogICAgbW9kZWwudGhyZXNob2xkID0gc2F2ZWRfdGhyZXNob2xkDQoNCmRlZiBydW5fc2luZ2xlX3ZhcmlhbnQodmFyaWFudF9uYW1lLCBNb2RlbENsYXNzLCBhcmdzLCBkb2NzX2JhdGNoLCBrZXlfaWR4KToNCiAgICBwcmludChmIlxuW3t2YXJpYW50X25hbWV9XSBUZXN0aW5nIE1lbW9yeSBSZXRlbnRpb24uLi4iKQ0KICAgIA0KICAgICMgMS4gQ29uZmlnIEhhbmRsaW5nDQogICAgIyBNQUMgdXNlcyAnY2h1bmtfc2l6ZScsIE1BRy9NQUwgdXNlICd3aW5kb3dfc2l6ZScNCiAgICBjb25maWcgPSB7DQogICAgICAgICJkaW0iOiBhcmdzLmRpbSwNCiAgICAgICAgImhpZGRlbl9kaW0iOiBhcmdzLmRpbSwNCiAgICAgICAgIm1lbW9yeV9kZXB0aCI6IDIsDQogICAgICAgICJudW1fcGVyc2lzdGVudF90b2tlbnMiOiAwLA0KICAgICAgICAidGhyZXNob2xkIjogMC4wDQogICAgfQ0KICAgIA0KICAgIGlmIHZhcmlhbnRfbmFtZSA9PSAnVGl0YW5NQUMnOg0KICAgICAgICBjb25maWdbImNodW5rX3NpemUiXSA9IDMyDQogICAgZWxzZToNCiAgICAgICAgY29uZmlnWyJ3aW5kb3dfc2l6ZSJdID0gMzINCiAgICAgICAgDQogICAgdHJ5Og0KICAgICAgICBtb2RlbF9iYXNlID0gTW9kZWxDbGFzcygqKmNvbmZpZykNCiAgICBleGNlcHQgRXhjZXB0aW9uIGFzIGU6DQogICAgICAgIHByaW50KGYiICBFcnJvciBpbnN0YW50aWF0aW5nIHt2YXJpYW50X25hbWV9OiB7ZX0iKQ0KICAgICAgICByZXR1cm4NCg0KICAgICMgW0NSSVRJQ0FMIFNURVBdIFByZS10cmFpbiB0aGUgJ0NvbnRleHQgRW5naW5lJyAoQXR0ZW50aW9uKQ0KICAgICMgT3RoZXJ3aXNlIEh5YnJpZCBSQUcgZmFpbHMgYmVjYXVzZSBSYW5kb20gV2VpZ2h0cyBkb24ndCBDb3B5Lg0KICAgIHByZXRyYWluX2lkZW50aXR5KG1vZGVsX2Jhc2UsIGFyZ3MuZGltKQ0KDQogICAgIyBbT3B0aW1pemF0aW9uIDIuMF0gSGlnaC1GaWRlbGl0eSBUdW5pbmcNCiAgICB3aXRoIHRvcmNoLm5vX2dyYWQoKToNCiAgICAgICAgbW9kZWxfYmFzZS5sdG0udGhldGEuZmlsbF8oMC4wMDEpIA0KICAgICAgICBtb2RlbF9iYXNlLmx0bS5hbHBoYS5maWxsXygwLjAwMDEpDQogICAgDQogICAgbW9kZWxfcmFnID0gVGl0YW5SQUcobW9kZWxfYmFzZSkNCiAgICANCiAgICAjIDIuIEJhc2VsaW5lIEV2YWx1YXRpb24NCiAgICBuZWVkbGVfZG9jID0gZG9jc19iYXRjaFtrZXlfaWR4XQ0KICAgIHRhcmdldCA9IG5lZWRsZV9kb2MNCiAgICBwcmVkX2luaXQgPSBtb2RlbF9yYWcudGl0YW4ubHRtLmZvcndhcmRfbm9fdXBkYXRlKHRhcmdldC5yZXNoYXBlKC0xLCBhcmdzLmRpbSkpDQogICAgZGlmZl9pbml0ID0gKHByZWRfaW5pdCAtIHRhcmdldC5yZXNoYXBlKC0xLCBhcmdzLmRpbSkpDQogICAgbG9zc19pbml0ID0gMC41KihkaWZmX2luaXQqKjIpLnN1bSgpLml0ZW0oKQ0KICAgIA0KICAgICMgMy4gRGlnZXN0aW9uDQogICAgIyBwcmludChmIiAgRGlnZXN0aW5nICgyMDAgZXBvY2hzIHBlciBkb2MpLi4uIikNCiAgICBmb3IgaSwgZG9jIGluIGVudW1lcmF0ZShkb2NzX2JhdGNoKToNCiAgICAgICAgZm9yIF8gaW4gcmFuZ2UoMjAwKTogDQogICAgICAgICAgICBtb2RlbF9yYWcuZGlnZXN0X2tub3dsZWRnZShkb2MpDQogICAgICAgICAgICANCiAgICAjIDQuIEZpbmFsIEV2YWx1YXRpb24NCiAgICBwcmVkX2ZpbmFsID0gbW9kZWxfcmFnLnRpdGFuLmx0bS5mb3J3YXJkX25vX3VwZGF0ZSh0YXJnZXQucmVzaGFwZSgtMSwgYXJncy5kaW0pKQ0KICAgIGRpZmZfZmluYWwgPSAocHJlZF9maW5hbCAtIHRhcmdldC5yZXNoYXBlKC0xLCBhcmdzLmRpbSkpDQogICAgbG9zc19hZnRlciA9IDAuNSooZGlmZl9maW5hbCoqMikuc3VtKCkuaXRlbSgpDQogICAgDQogICAgIyA1LiBNZXRyaWNzDQogICAgdG90YWxfZWxlbWVudHMgPSBhcmdzLmRvY19sZW4gKiBhcmdzLmRpbQ0KICAgIHNpZ25hbF9lbmVyZ3kgPSAoKHRhcmdldC5yZXNoYXBlKC0xLCBhcmdzLmRpbSkpKioyKS5zdW0oKS5pdGVtKCkNCiAgICBlcnJvcl9lbmVyZ3kgPSAyICogbG9zc19hZnRlcg0KICAgIA0KICAgIGZpZGVsaXR5ID0gMS4wIC0gKGVycm9yX2VuZXJneSAvIChzaWduYWxfZW5lcmd5ICsgMWUtOSkpDQogICAgZmlkZWxpdHkgPSBtYXgoMC4wLCBmaWRlbGl0eSkgKiAxMDAuMA0KICAgIA0KICAgIHByaW50KGYiICBJbml0aWFsIExvc3M6IHtsb3NzX2luaXQ6LjRmfSIpDQogICAgcHJpbnQoZiIgIEZpbmFsIExvc3M6ICAge2xvc3NfYWZ0ZXI6LjRmfSIpDQogICAgcHJpbnQoZiIgIEZpZGVsaXR5OiAgICAge2ZpZGVsaXR5Oi4yZn0lIikNCiAgICANCiAgICBpZiBsb3NzX2FmdGVyIDwgbG9zc19pbml0Og0KICAgICAgICBwcmludChmIiAgUmVzdWx0OiBQQVNTIChJbXByb3ZlbWVudCB7KChsb3NzX2luaXQtbG9zc19hZnRlcikvbG9zc19pbml0KSoxMDA6LjFmfSUpIikNCiAgICBlbHNlOg0KICAgICAgICBwcmludChmIiAgUmVzdWx0OiBGQUlMIikNCiAgICAgICAgDQogICAgIyAtLS0gTWV0aG9kIDQ6IEh5YnJpZCBSQUcgKFRoZSBCZXN0IG9mIEJvdGggV29ybGRzKSAtLS0NCiAgICBwcmludChmIlxuW3t2YXJpYW50X25hbWV9XSBUZXN0aW5nIEh5YnJpZCBSQUcgKFdlaWdodCArIENvbnRleHQpLi4uIikNCiAgICAjIFNjZW5hcmlvOiBOZWVkbGUgaXMgaW4gQ29udGV4dCwgTm9pc2UgaXMgaW4gV2VpZ2h0cy4NCiAgICANCiAgICB3aXRoIHRvcmNoLm5vX2dyYWQoKToNCiAgICAgICAgIyBJbnB1dDogTmVlZGxlIERvYyAoQmF0Y2g9MSwgTGVuPWRvY19sZW4sIERpbT1kaW0pDQogICAgICAgICMgQnkgcGFzc2luZyBpdCBhcyBpbnB1dCAneCcgdG8gdGhlIG1vZGVsIGZvcndhcmQgcGFzcywgDQogICAgICAgICMgdGhlIEF0dGVudGlvbiBtZWNoYW5pc20gc2VlcyBpdCBkaXJlY3RseS4NCiAgICAgICAgaW5wID0gbmVlZGxlX2RvYyAjIEFscmVhZHkgaGFzIGJhdGNoIGRpbSBbMSwgMzIsIDY0XQ0KICAgICAgICBvdXRwdXQgPSBtb2RlbF9yYWcoaW5wKQ0KICAgICAgICANCiAgICBkaWZmX2h5YnJpZCA9IChvdXRwdXQgLSBpbnApDQogICAgbG9zc19oeWJyaWQgPSAwLjUqKGRpZmZfaHlicmlkKioyKS5zdW0oKS5pdGVtKCkNCiAgICANCiAgICBlcnJvcl9lbmVyZ3lfaHliID0gMiAqIGxvc3NfaHlicmlkDQogICAgZmlkZWxpdHlfaHliID0gMS4wIC0gKGVycm9yX2VuZXJneV9oeWIgLyAoc2lnbmFsX2VuZXJneSArIDFlLTkpKQ0KICAgIGZpZGVsaXR5X2h5YiA9IG1heCgwLjAsIGZpZGVsaXR5X2h5YikgKiAxMDAuMA0KICAgIA0KICAgIHByaW50KGYiICBIeWJyaWQgRmlkZWxpdHk6IHtmaWRlbGl0eV9oeWI6LjJmfSUgKFNob3VsZCBiZSB+MTAwJSkiKQ0KICAgIGlmIGZpZGVsaXR5X2h5YiA+IDk5LjA6DQogICAgICAgIHByaW50KGYiICBSZXN1bHQ6IFBBU1MgKFBlcmZlY3QgUmVjYWxsIHZpYSBBdHRlbnRpb24pIikNCg0KZGVmIGNvbXBhcmVfbWV0aG9kcyhhcmdzKToNCiAgICBwcmludCgiLS0tIENvbmR1Y3RpbmcgUkFHIENvbXBhcmlzb24gRXhwZXJpbWVudCAoQWxsIFZhcmlhbnRzKSAtLS0iKQ0KICAgIHByaW50KGYiRG9jczoge2FyZ3MubnVtX2RvY3N9LCBEb2MgTGVuOiB7YXJncy5kb2NfbGVufSwgRGltOiB7YXJncy5kaW19IikNCiAgICANCiAgICAjIDEuIFNldHVwIERhdGENCiAgICBkb2NzLCBfLCBrZXlfaWR4ID0gZ2VuZXJhdGVfaGF5c3RhY2soYXJncy5udW1fZG9jcywgYXJncy5kb2NfbGVuLCBhcmdzLmRpbSkNCiAgICBkb2NzX2JhdGNoID0gW2QudW5zcXVlZXplKDApIGZvciBkIGluIGRvY3NdDQogICAgDQogICAgIyAyLiBUcmFkaXRpb25hbCBSQUcgQmFzZWxpbmUgKE9ubHkgZm9yIE1BQywgYXMgY29udGV4dCBjb25jYXQgaXMgdW5pcXVlIHRvIGl0IGNvbmNlcHR1YWxseSkNCiAgICBwcmludCgiXG5bQmFzZWxpbmU6IFRyYWRpdGlvbmFsIFJBRyAoQ29uY2F0ZW5hdGlvbildIikNCiAgICAjIE9ubHkgTUFDIGlzIHJlYWxseSBzdWl0YWJsZSBmb3IgJ0NvbmNhdCBhbGwnIG5haXZlIFJBRywgb3RoZXJzIHVzZSBzbGlkaW5nIHdpbmRvdy9nYXRpbmcuDQogICAgIyBCdXQgZm9yIE9PTSB0ZXN0aW5nLCBhbnkgbW9kZWwgcmVjZWl2aW5nIGh1Z2UgaW5wdXQgd29ya3MuDQogICAgdHJ5Og0KICAgICAgICBmdWxsX2NvbnRleHQgPSB0b3JjaC5jYXQoZG9jc19iYXRjaCwgZGltPTEpDQogICAgICAgIGlmIGZ1bGxfY29udGV4dC5zaXplKDEpID4gMjA0ODoNCiAgICAgICAgICAgIHByaW50KGYiICBSZXN1bHQ6IEZBSUwgKENvbnRleHQgTGVuZ3RoIHtmdWxsX2NvbnRleHQuc2l6ZSgxKX0gPiAyMDQ4KSIpDQogICAgICAgIGVsc2U6DQogICAgICAgICAgICBwcmludChmIiAgUmVzdWx0OiBQQVNTIChQcm9jZXNzZWQge2Z1bGxfY29udGV4dC5zaXplKDEpfSB0b2tlbnMpIikNCiAgICBleGNlcHQgRXhjZXB0aW9uIGFzIGU6DQogICAgICAgICBwcmludChmIiAgUmVzdWx0OiBGQUlMICh7ZX0pIikNCg0KICAgICMgMy4gVGl0YW5SQUcgVmFyaWFudHMNCiAgICB2YXJpYW50cyA9IFsNCiAgICAgICAgKCdUaXRhbk1BQycsIFRpdGFuTUFDKSwNCiAgICAgICAgKCdUaXRhbk1BRycsIFRpdGFuTUFHKSwNCiAgICAgICAgKCdUaXRhbk1BTCcsIFRpdGFuTUFMKQ0KICAgIF0NCiAgICANCiAgICBmb3IgbmFtZSwgY2xzIGluIHZhcmlhbnRzOg0KICAgICAgICBydW5fc2luZ2xlX3ZhcmlhbnQobmFtZSwgY2xzLCBhcmdzLCBkb2NzX2JhdGNoLCBrZXlfaWR4KQ0KDQppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOg0KICAgIHBhcnNlciA9IGFyZ3BhcnNlLkFyZ3VtZW50UGFyc2VyKCkNCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLW51bV9kb2NzIiwgdHlwZT1pbnQsIGRlZmF1bHQ9NTAsIGhlbHA9Ik51bWJlciBvZiByZXRyaWV2ZWQgZG9jdW1lbnRzIikNCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLWRvY19sZW4iLCB0eXBlPWludCwgZGVmYXVsdD02NCwgaGVscD0iTGVuZ3RoIG9mIGVhY2ggZG9jdW1lbnQiKQ0KICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tZGltIiwgdHlwZT1pbnQsIGRlZmF1bHQ9NjQpDQogICAgYXJncyA9IHBhcnNlci5wYXJzZV9hcmdzKCkNCiAgICANCiAgICBjb21wYXJlX21ldGhvZHMoYXJncykNCg==',
    'data/sample_essays.py': 'IiIiDQpTYW1wbGUgRXNzYXlzIGZvciBUaXRhblJBRyBFeHBlcmltZW50cw0KDQpUaGVzZSBlc3NheXMgY29udGFpbiBmYWN0dWFsIGluZm9ybWF0aW9uIHRoYXQgY2FuIGJlIHVzZWQgdG8gdGVzdA0KdGhlIG1vZGVsJ3MgYWJpbGl0eSB0byBtZW1vcml6ZSBhbmQgcmVjYWxsIHNwZWNpZmljIGRldGFpbHMuDQoiIiINCg0KRVNTQVlfQ0xJTUFURSA9ICIiIg0KIyBUaGUgU2NpZW5jZSBvZiBDbGltYXRlIENoYW5nZQ0KDQojIyBJbnRyb2R1Y3Rpb24NCkNsaW1hdGUgY2hhbmdlIHJlZmVycyB0byBsb25nLXRlcm0gc2hpZnRzIGluIGdsb2JhbCB0ZW1wZXJhdHVyZXMgYW5kIHdlYXRoZXIgcGF0dGVybnMuIA0KV2hpbGUgbmF0dXJhbCBmYWN0b3JzIGxpa2Ugdm9sY2FuaWMgZXJ1cHRpb25zIGFuZCBzb2xhciBjeWNsZXMgY2FuIGluZmx1ZW5jZSBjbGltYXRlLCANCmh1bWFuIGFjdGl2aXRpZXMgaGF2ZSBiZWVuIHRoZSBwcmltYXJ5IGRyaXZlciBzaW5jZSB0aGUgMTgwMHMuDQoNCiMjIEtleSBGYWN0cw0KLSBUaGUgZ2xvYmFsIGF2ZXJhZ2UgdGVtcGVyYXR1cmUgaGFzIHJpc2VuIGJ5IGFwcHJveGltYXRlbHkgMS4xwrBDIHNpbmNlIHByZS1pbmR1c3RyaWFsIHRpbWVzLg0KLSBUaGUgY29uY2VudHJhdGlvbiBvZiBDTzIgaW4gdGhlIGF0bW9zcGhlcmUgcmVhY2hlZCA0MjEgcGFydHMgcGVyIG1pbGxpb24gKHBwbSkgaW4gMjAyMy4NCi0gU2VhIGxldmVscyBoYXZlIHJpc2VuIGJ5IGFib3V0IDIwIGNlbnRpbWV0ZXJzICg4IGluY2hlcykgc2luY2UgMTkwMC4NCi0gVGhlIEFyY3RpYyBpcyB3YXJtaW5nIG5lYXJseSBmb3VyIHRpbWVzIGZhc3RlciB0aGFuIHRoZSBnbG9iYWwgYXZlcmFnZS4NCg0KIyMgQ2F1c2VzDQpUaGUgYnVybmluZyBvZiBmb3NzaWwgZnVlbHPigJRjb2FsLCBvaWwsIGFuZCBuYXR1cmFsIGdhc+KAlHJlbGVhc2VzIGNhcmJvbiBkaW94aWRlIChDTzIpIA0KYW5kIG90aGVyIGdyZWVuaG91c2UgZ2FzZXMgaW50byB0aGUgYXRtb3NwaGVyZS4gVGhlc2UgZ2FzZXMgdHJhcCBoZWF0IGZyb20gdGhlIHN1biwgDQpjcmVhdGluZyBhICJncmVlbmhvdXNlIGVmZmVjdCIgdGhhdCB3YXJtcyB0aGUgcGxhbmV0Lg0KDQojIyBJbXBhY3RzDQoxLiAqKkV4dHJlbWUgV2VhdGhlcioqOiBNb3JlIGZyZXF1ZW50IGhlYXR3YXZlcywgZHJvdWdodHMsIGFuZCBpbnRlbnNlIHN0b3Jtcy4NCjIuICoqRWNvc3lzdGVtIERpc3J1cHRpb24qKjogQ29yYWwgYmxlYWNoaW5nLCBzcGVjaWVzIG1pZ3JhdGlvbiwgYW5kIGJpb2RpdmVyc2l0eSBsb3NzLg0KMy4gKipIdW1hbiBIZWFsdGgqKjogSW5jcmVhc2VkIGhlYXQtcmVsYXRlZCBpbGxuZXNzZXMgYW5kIHNwcmVhZCBvZiB2ZWN0b3ItYm9ybmUgZGlzZWFzZXMuDQo0LiAqKkFncmljdWx0dXJlKio6IFNoaWZ0cyBpbiBncm93aW5nIHNlYXNvbnMgYW5kIHJlZHVjZWQgY3JvcCB5aWVsZHMgaW4gc29tZSByZWdpb25zLg0KDQojIyBTb2x1dGlvbnMNCi0gVHJhbnNpdGlvbiB0byByZW5ld2FibGUgZW5lcmd5IHNvdXJjZXMgKHNvbGFyLCB3aW5kLCBoeWRybykuDQotIEltcHJvdmUgZW5lcmd5IGVmZmljaWVuY3kgaW4gYnVpbGRpbmdzIGFuZCB0cmFuc3BvcnRhdGlvbi4NCi0gUHJvdGVjdCBhbmQgcmVzdG9yZSBmb3Jlc3RzLCB3aGljaCBhYnNvcmIgQ08yLg0KLSBJbXBsZW1lbnQgY2FyYm9uIHByaWNpbmcgdG8gaW5jZW50aXZpemUgZW1pc3Npb24gcmVkdWN0aW9ucy4NCg0KIyMgQ29uY2x1c2lvbg0KQWRkcmVzc2luZyBjbGltYXRlIGNoYW5nZSByZXF1aXJlcyB1cmdlbnQsIGNvb3JkaW5hdGVkIGdsb2JhbCBhY3Rpb24uIFRoZSBQYXJpcyBBZ3JlZW1lbnQgDQpvZiAyMDE1IHNldCB0aGUgZ29hbCBvZiBsaW1pdGluZyB3YXJtaW5nIHRvIDEuNcKwQyBhYm92ZSBwcmUtaW5kdXN0cmlhbCBsZXZlbHMuDQoiIiINCg0KRVNTQVlfQUkgPSAiIiINCiMgVGhlIEV2b2x1dGlvbiBvZiBBcnRpZmljaWFsIEludGVsbGlnZW5jZQ0KDQojIyBIaXN0b3JpY2FsIEJhY2tncm91bmQNClRoZSB0ZXJtICJBcnRpZmljaWFsIEludGVsbGlnZW5jZSIgd2FzIGNvaW5lZCBieSBKb2huIE1jQ2FydGh5IGluIDE5NTYgYXQgdGhlIERhcnRtb3V0aCANCkNvbmZlcmVuY2UuIEVhcmx5IEFJIHJlc2VhcmNoIGZvY3VzZWQgb24gc3ltYm9saWMgcmVhc29uaW5nIGFuZCBwcm9ibGVtLXNvbHZpbmcuDQoNCiMjIFRpbWVsaW5lIG9mIE1ham9yIE1pbGVzdG9uZXMNCi0gKioxOTUwKio6IEFsYW4gVHVyaW5nIHByb3Bvc2VzIHRoZSAiVHVyaW5nIFRlc3QiIGZvciBtYWNoaW5lIGludGVsbGlnZW5jZS4NCi0gKioxOTU2Kio6IERhcnRtb3V0aCBDb25mZXJlbmNlIG1hcmtzIHRoZSBiaXJ0aCBvZiBBSSBhcyBhIGZpZWxkLg0KLSAqKjE5OTcqKjogSUJNJ3MgRGVlcCBCbHVlIGRlZmVhdHMgd29ybGQgY2hlc3MgY2hhbXBpb24gR2FycnkgS2FzcGFyb3YuDQotICoqMjAxMSoqOiBJQk0gV2F0c29uIHdpbnMgSmVvcGFyZHkhIGFnYWluc3QgaHVtYW4gY2hhbXBpb25zLg0KLSAqKjIwMTYqKjogR29vZ2xlJ3MgQWxwaGFHbyBkZWZlYXRzIExlZSBTZWRvbCBpbiB0aGUgZ2FtZSBvZiBHby4NCi0gKioyMDIyKio6IENoYXRHUFQgbGF1bmNoZXMsIGRlbW9uc3RyYXRpbmcgYWR2YW5jZWQgY29udmVyc2F0aW9uYWwgQUkuDQoNCiMjIFR5cGVzIG9mIEFJDQoxLiAqKk5hcnJvdyBBSSAoQU5JKSoqOiBEZXNpZ25lZCBmb3Igc3BlY2lmaWMgdGFza3MgKGUuZy4sIGltYWdlIHJlY29nbml0aW9uLCBsYW5ndWFnZSB0cmFuc2xhdGlvbikuDQoyLiAqKkdlbmVyYWwgQUkgKEFHSSkqKjogSHlwb3RoZXRpY2FsIEFJIHdpdGggaHVtYW4tbGV2ZWwgY29nbml0aXZlIGFiaWxpdGllcyBhY3Jvc3MgYWxsIGRvbWFpbnMuDQozLiAqKlN1cGVyaW50ZWxsaWdlbnQgQUkgKEFTSSkqKjogVGhlb3JldGljYWwgQUkgc3VycGFzc2luZyBodW1hbiBpbnRlbGxpZ2VuY2UgZW50aXJlbHkuDQoNCiMjIEN1cnJlbnQgQXBwbGljYXRpb25zDQotICoqSGVhbHRoY2FyZSoqOiBNZWRpY2FsIGltYWdpbmcgYW5hbHlzaXMsIGRydWcgZGlzY292ZXJ5LCBwZXJzb25hbGl6ZWQgdHJlYXRtZW50IHBsYW5zLg0KLSAqKkZpbmFuY2UqKjogRnJhdWQgZGV0ZWN0aW9uLCBhbGdvcml0aG1pYyB0cmFkaW5nLCBjcmVkaXQgc2NvcmluZy4NCi0gKipUcmFuc3BvcnRhdGlvbioqOiBBdXRvbm9tb3VzIHZlaGljbGVzLCB0cmFmZmljIG9wdGltaXphdGlvbiwgcm91dGUgcGxhbm5pbmcuDQotICoqRWR1Y2F0aW9uKio6IFBlcnNvbmFsaXplZCBsZWFybmluZywgYXV0b21hdGVkIGdyYWRpbmcsIHR1dG9yaW5nIHN5c3RlbXMuDQoNCiMjIEtleSBUZWNobm9sb2dpZXMNClRoZSBtb2Rlcm4gQUkgcmV2b2x1dGlvbiBpcyBidWlsdCBvbjoNCi0gKipEZWVwIExlYXJuaW5nKio6IE5ldXJhbCBuZXR3b3JrcyB3aXRoIG11bHRpcGxlIGxheWVycyBmb3IgZmVhdHVyZSBleHRyYWN0aW9uLg0KLSAqKlRyYW5zZm9ybWVycyoqOiBBcmNoaXRlY3R1cmUgYmVoaW5kIGxhcmdlIGxhbmd1YWdlIG1vZGVscyBsaWtlIEdQVC00IGFuZCBDbGF1ZGUuDQotICoqUmVpbmZvcmNlbWVudCBMZWFybmluZyoqOiBUcmFpbmluZyBhZ2VudHMgdGhyb3VnaCByZXdhcmQtYmFzZWQgZmVlZGJhY2suDQoNCiMjIEV0aGljYWwgQ29uc2lkZXJhdGlvbnMNCi0gQmlhcyBpbiB0cmFpbmluZyBkYXRhIGxlYWRpbmcgdG8gdW5mYWlyIG91dGNvbWVzLg0KLSBKb2IgZGlzcGxhY2VtZW50IGFuZCBlY29ub21pYyBpbmVxdWFsaXR5Lg0KLSBQcml2YWN5IGNvbmNlcm5zIGZyb20gZGF0YSBjb2xsZWN0aW9uLg0KLSBQb3RlbnRpYWwgbWlzdXNlIGZvciBtaXNpbmZvcm1hdGlvbiBhbmQgc3VydmVpbGxhbmNlLg0KDQojIyBUaGUgRnV0dXJlDQpFeHBlcnRzIHByZWRpY3QgQUkgd2lsbCBjb250aW51ZSB0byB0cmFuc2Zvcm0gaW5kdXN0cmllcywgd2l0aCBlc3RpbWF0ZWQgZWNvbm9taWMgaW1wYWN0IA0Kb2YgJDE1LjcgdHJpbGxpb24gYnkgMjAzMCBhY2NvcmRpbmcgdG8gUHdDLiBUaGUgZGV2ZWxvcG1lbnQgb2YgQUdJIHJlbWFpbnMgYSBsb25nLXRlcm0gZ29hbC4NCiIiIg0KDQpFU1NBWV9TUEFDRSA9ICIiIg0KIyBFeHBsb3JpbmcgdGhlIFNvbGFyIFN5c3RlbQ0KDQojIyBPdXIgQ29zbWljIE5laWdoYm9yaG9vZA0KVGhlIFNvbGFyIFN5c3RlbSBjb25zaXN0cyBvZiB0aGUgU3VuLCBlaWdodCBwbGFuZXRzLCBkd2FyZiBwbGFuZXRzLCBtb29ucywgYXN0ZXJvaWRzLCANCmFuZCBjb21ldHMuIEl0IGZvcm1lZCBhcHByb3hpbWF0ZWx5IDQuNiBiaWxsaW9uIHllYXJzIGFnbyBmcm9tIGEgZ2lhbnQgbW9sZWN1bGFyIGNsb3VkLg0KDQojIyBUaGUgU3VuDQotICoqVHlwZSoqOiBHLXR5cGUgbWFpbi1zZXF1ZW5jZSBzdGFyICh5ZWxsb3cgZHdhcmYpDQotICoqRGlhbWV0ZXIqKjogMSwzOTIsNzAwIGttICgxMDkgdGltZXMgRWFydGgncyBkaWFtZXRlcikNCi0gKipTdXJmYWNlIFRlbXBlcmF0dXJlKio6IDUsNTAwwrBDICg5LDkwMMKwRikNCi0gKipDb3JlIFRlbXBlcmF0dXJlKio6IDE1IG1pbGxpb27CsEMNCi0gKipEaXN0YW5jZSBmcm9tIEVhcnRoKio6IDE0OS42IG1pbGxpb24ga20gKDEgQXN0cm9ub21pY2FsIFVuaXQpDQoNCiMjIFRoZSBQbGFuZXRzDQojIyMgSW5uZXIgUGxhbmV0cyAoUm9ja3kvVGVycmVzdHJpYWwpDQoxLiAqKk1lcmN1cnkqKjogU21hbGxlc3QgcGxhbmV0LCA4OC1kYXkgb3JiaXQsIG5vIG1vb25zLg0KMi4gKipWZW51cyoqOiBIb3R0ZXN0IHBsYW5ldCAoNDc1wrBDIHN1cmZhY2UpLCByb3RhdGVzIGJhY2t3YXJkcy4NCjMuICoqRWFydGgqKjogT25seSBrbm93biBwbGFuZXQgd2l0aCBsaXF1aWQgd2F0ZXIgb24gc3VyZmFjZSBhbmQgbGlmZS4NCjQuICoqTWFycyoqOiAiUmVkIFBsYW5ldCIsIGhhcyB0aGUgbGFyZ2VzdCB2b2xjYW5vIChPbHltcHVzIE1vbnMsIDIxLjkga20gaGlnaCkuDQoNCiMjIyBPdXRlciBQbGFuZXRzIChHYXMvSWNlIEdpYW50cykNCjUuICoqSnVwaXRlcioqOiBMYXJnZXN0IHBsYW5ldCAoMTF4IEVhcnRoJ3MgZGlhbWV0ZXIpLCBHcmVhdCBSZWQgU3BvdCBzdG9ybS4NCjYuICoqU2F0dXJuKio6IEZhbW91cyBmb3IgaXRzIHJpbmcgc3lzdGVtLCA4MyBrbm93biBtb29ucyBpbmNsdWRpbmcgVGl0YW4uDQo3LiAqKlVyYW51cyoqOiBSb3RhdGVzIG9uIGl0cyBzaWRlICg5OMKwIGF4aWFsIHRpbHQpLCAyNyBtb29ucy4NCjguICoqTmVwdHVuZSoqOiBTdHJvbmdlc3Qgd2luZHMgaW4gdGhlIFNvbGFyIFN5c3RlbSAoMiwxMDAga20vaCkuDQoNCiMjIE5vdGFibGUgTWlzc2lvbnMNCi0gKipWb3lhZ2VyIDEgJiAyICgxOTc3KSoqOiBGaXJzdCBzcGFjZWNyYWZ0IHRvIHJlYWNoIGludGVyc3RlbGxhciBzcGFjZS4NCi0gKipIdWJibGUgU3BhY2UgVGVsZXNjb3BlICgxOTkwKSoqOiBSZXZvbHV0aW9uaXplZCBvdXIgdmlldyBvZiB0aGUgdW5pdmVyc2UuDQotICoqTWFycyBSb3ZlcnMqKjogU3Bpcml0LCBPcHBvcnR1bml0eSwgQ3VyaW9zaXR5LCBQZXJzZXZlcmFuY2UgZXhwbG9yZWQgTWFycy4NCi0gKipKYW1lcyBXZWJiIFNwYWNlIFRlbGVzY29wZSAoMjAyMSkqKjogTW9zdCBwb3dlcmZ1bCBzcGFjZSB0ZWxlc2NvcGUgZXZlciBidWlsdC4NCg0KIyMgRHdhcmYgUGxhbmV0cw0KUGx1dG8gd2FzIHJlY2xhc3NpZmllZCBhcyBhIGR3YXJmIHBsYW5ldCBpbiAyMDA2IGJ5IHRoZSBJbnRlcm5hdGlvbmFsIEFzdHJvbm9taWNhbCBVbmlvbi4gDQpPdGhlciBkd2FyZiBwbGFuZXRzIGluY2x1ZGUgRXJpcywgSGF1bWVhLCBNYWtlbWFrZSwgYW5kIENlcmVzLg0KDQojIyBGdXR1cmUgb2YgU3BhY2UgRXhwbG9yYXRpb24NCi0gKioyMDI0LTIwMjUqKjogQXJ0ZW1pcyBwcm9ncmFtIGFpbXMgdG8gcmV0dXJuIGh1bWFucyB0byB0aGUgTW9vbi4NCi0gKioyMDMwcyoqOiBOQVNBIGFuZCBTcGFjZVggcGxhbm5pbmcgY3Jld2VkIG1pc3Npb25zIHRvIE1hcnMuDQotICoqTG9uZy10ZXJtKio6IFBvdGVudGlhbCBjb2xvbml6YXRpb24gb2YgTWFycyBhbmQgYXN0ZXJvaWQgbWluaW5nLg0KIiIiDQoNCiMgUXVlc3Rpb25zIGZvciB0ZXN0aW5nICh3aXRoIGV4cGVjdGVkIGFuc3dlcnMgZnJvbSB0aGUgZXNzYXlzKQ0KVEVTVF9RVUVTVElPTlMgPSB7DQogICAgImNsaW1hdGUiOiBbDQogICAgICAgICgiV2hhdCBpcyB0aGUgY3VycmVudCBDTzIgY29uY2VudHJhdGlvbiBpbiB0aGUgYXRtb3NwaGVyZT8iLCAiNDIxIHBhcnRzIHBlciBtaWxsaW9uIChwcG0pIGluIDIwMjMiKSwNCiAgICAgICAgKCJIb3cgbXVjaCBoYXMgdGhlIGdsb2JhbCB0ZW1wZXJhdHVyZSByaXNlbiBzaW5jZSBwcmUtaW5kdXN0cmlhbCB0aW1lcz8iLCAiYXBwcm94aW1hdGVseSAxLjHCsEMiKSwNCiAgICAgICAgKCJXaGF0IHllYXIgd2FzIHRoZSBQYXJpcyBBZ3JlZW1lbnQgc2lnbmVkPyIsICIyMDE1IiksDQogICAgICAgICgiSG93IG11Y2ggaGF2ZSBzZWEgbGV2ZWxzIHJpc2VuIHNpbmNlIDE5MDA/IiwgImFib3V0IDIwIGNlbnRpbWV0ZXJzICg4IGluY2hlcykiKSwNCiAgICBdLA0KICAgICJhaSI6IFsNCiAgICAgICAgKCJXaG8gY29pbmVkIHRoZSB0ZXJtICdBcnRpZmljaWFsIEludGVsbGlnZW5jZSc/IiwgIkpvaG4gTWNDYXJ0aHkgaW4gMTk1NiIpLA0KICAgICAgICAoIldoZW4gZGlkIERlZXAgQmx1ZSBkZWZlYXQgR2FycnkgS2FzcGFyb3Y/IiwgIjE5OTciKSwNCiAgICAgICAgKCJXaGF0IGlzIHRoZSBlc3RpbWF0ZWQgZWNvbm9taWMgaW1wYWN0IG9mIEFJIGJ5IDIwMzA/IiwgIiQxNS43IHRyaWxsaW9uIGFjY29yZGluZyB0byBQd0MiKSwNCiAgICAgICAgKCJXaGF0IGFyY2hpdGVjdHVyZSBwb3dlcnMgR1BULTQgYW5kIENsYXVkZT8iLCAiVHJhbnNmb3JtZXJzIiksDQogICAgXSwNCiAgICAic3BhY2UiOiBbDQogICAgICAgICgiSG93IG9sZCBpcyB0aGUgU29sYXIgU3lzdGVtPyIsICJhcHByb3hpbWF0ZWx5IDQuNiBiaWxsaW9uIHllYXJzIiksDQogICAgICAgICgiV2hhdCBpcyB0aGUgdGFsbGVzdCB2b2xjYW5vIGluIHRoZSBTb2xhciBTeXN0ZW0/IiwgIk9seW1wdXMgTW9ucywgMjEuOSBrbSBoaWdoIiksDQogICAgICAgICgiV2hlbiB3YXMgdGhlIEphbWVzIFdlYmIgU3BhY2UgVGVsZXNjb3BlIGxhdW5jaGVkPyIsICIyMDIxIiksDQogICAgICAgICgiV2hhdCB5ZWFyIHdhcyBQbHV0byByZWNsYXNzaWZpZWQgYXMgYSBkd2FyZiBwbGFuZXQ/IiwgIjIwMDYiKSwNCiAgICBdLA0KfQ0KDQpkZWYgZ2V0X2FsbF9lc3NheXMoKToNCiAgICAiIiJSZXR1cm4gYWxsIGVzc2F5cyBhcyBhIGRpY3Rpb25hcnkuIiIiDQogICAgcmV0dXJuIHsNCiAgICAgICAgImNsaW1hdGUiOiBFU1NBWV9DTElNQVRFLA0KICAgICAgICAiYWkiOiBFU1NBWV9BSSwNCiAgICAgICAgInNwYWNlIjogRVNTQVlfU1BBQ0UsDQogICAgfQ0KDQpkZWYgZ2V0X3Rlc3RfcXVlc3Rpb25zKCk6DQogICAgIiIiUmV0dXJuIHRlc3QgcXVlc3Rpb25zIHdpdGggZXhwZWN0ZWQgYW5zd2Vycy4iIiINCiAgICByZXR1cm4gVEVTVF9RVUVTVElPTlMNCg==',
    'projects/hybrid_titans/essay_rag_demo.py': '...',
    'projects/hybrid_titans/hybrid_titan_demo.py': 'IiIiDQpIeWJyaWQgVGl0YW4gRGVtbzogVGl0YW5SQUcgUmV0cmlldmFsICsgRkxBTi1UNSBHZW5lcmF0aW9uDQoNClRoaXMgZXhwZXJpbWVudCBkZW1vbnN0cmF0ZXMgRGlyZWN0aW9uIEEgZnJvbSB0aGUgcmVzZWFyY2ggcGxhbjoNCi0gVGl0YW5SQUc6IFBlcmZvcm1zIHNlbWFudGljIHJldHJpZXZhbCB1c2luZyBNZW1vcnkgKyBFbnNlbWJsZSBGdXNpb24NCi0gRmxhbi1UNS0yOiBHZW5lcmF0ZXMgbmF0dXJhbCBsYW5ndWFnZSBhbnN3ZXJzIGJhc2VkIG9uIHJldHJpZXZlZCBjb250ZXh0DQoNCkFyY2hpdGVjdHVyZToNCiAgICBRdWVzdGlvbiDihpIgVGl0YW5SQUcg4oaSIENvbnRleHQg4oaSIEZMQU4tVDUtTGFyZ2Ug4oaSIEFuc3dlcg0KIiIiDQoNCmltcG9ydCB0b3JjaA0KaW1wb3J0IHRvcmNoLm5uIGFzIG5uDQppbXBvcnQgYXJncGFyc2UNCmltcG9ydCBzeXMNCmltcG9ydCBvcw0KDQojIEFkZCBwYXJlbnQgZGlyZWN0b3J5IHRvIHBhdGggKGh5YnJpZF90aXRhbnMgLT4gcHJvamVjdHMgLT4gVGl0YW5MTE0pDQpzeXMucGF0aC5pbnNlcnQoMCwgb3MucGF0aC5kaXJuYW1lKG9zLnBhdGguZGlybmFtZShvcy5wYXRoLmRpcm5hbWUob3MucGF0aC5hYnNwYXRoKF9fZmlsZV9fKSkpKSkNCnN5cy5wYXRoLmluc2VydCgwLCBvcy5wYXRoLmpvaW4ob3MucGF0aC5kaXJuYW1lKG9zLnBhdGguZGlybmFtZShvcy5wYXRoLmRpcm5hbWUob3MucGF0aC5hYnNwYXRoKF9fZmlsZV9fKSkpKSwgJ3NyYycpKQ0KDQpmcm9tIG1haW4gaW1wb3J0IFRpdGFuUkFHDQpmcm9tIGRhdGEuc2FtcGxlX2Vzc2F5cyBpbXBvcnQgRVNTQVlfQ0xJTUFURSwgRVNTQVlfQUksIEVTU0FZX1NQQUNFLCBURVNUX1FVRVNUSU9OUw0KDQoNCmNsYXNzIFNlbnRlbmNlVHJhbnNmb3JtZXJFbWJlZGRlcihubi5Nb2R1bGUpOg0KICAgICIiIlNlbWFudGljIGVtYmVkZGVyIHVzaW5nIHNlbnRlbmNlLXRyYW5zZm9ybWVycy4iIiINCiAgICANCiAgICBkZWYgX19pbml0X18oc2VsZiwgbW9kZWxfbmFtZT0iYWxsLU1pbmlMTS1MNi12MiIsIHRhcmdldF9kaW09MjU2KToNCiAgICAgICAgc3VwZXIoKS5fX2luaXRfXygpDQogICAgICAgIGZyb20gc2VudGVuY2VfdHJhbnNmb3JtZXJzIGltcG9ydCBTZW50ZW5jZVRyYW5zZm9ybWVyDQogICAgICAgIHNlbGYubW9kZWwgPSBTZW50ZW5jZVRyYW5zZm9ybWVyKG1vZGVsX25hbWUpDQogICAgICAgIHNlbGYuc291cmNlX2RpbSA9IHNlbGYubW9kZWwuZ2V0X3NlbnRlbmNlX2VtYmVkZGluZ19kaW1lbnNpb24oKQ0KICAgICAgICBzZWxmLnRhcmdldF9kaW0gPSB0YXJnZXRfZGltDQogICAgICAgIA0KICAgICAgICBpZiBzZWxmLnNvdXJjZV9kaW0gIT0gdGFyZ2V0X2RpbToNCiAgICAgICAgICAgIHNlbGYucHJvamVjdGlvbiA9IG5uLkxpbmVhcihzZWxmLnNvdXJjZV9kaW0sIHRhcmdldF9kaW0pDQogICAgICAgIGVsc2U6DQogICAgICAgICAgICBzZWxmLnByb2plY3Rpb24gPSBOb25lDQogICAgICAgIA0KICAgICAgICBwcmludChmIuKchSBMb2FkZWQgU2VudGVuY2VUcmFuc2Zvcm1lcjoge21vZGVsX25hbWV9IChkaW09e3NlbGYuc291cmNlX2RpbX0pIikNCiAgICANCiAgICBkZWYgZm9yd2FyZChzZWxmLCB0ZXh0KToNCiAgICAgICAgd2l0aCB0b3JjaC5ub19ncmFkKCk6DQogICAgICAgICAgICBlbWJlZGRpbmcgPSBzZWxmLm1vZGVsLmVuY29kZSh0ZXh0LCBjb252ZXJ0X3RvX3RlbnNvcj1UcnVlKQ0KICAgICAgICAgICAgZW1iZWRkaW5nID0gZW1iZWRkaW5nLmNsb25lKCkuY3B1KCkNCiAgICAgICAgDQogICAgICAgIGlmIHNlbGYucHJvamVjdGlvbiBpcyBub3QgTm9uZToNCiAgICAgICAgICAgIHdpdGggdG9yY2gubm9fZ3JhZCgpOg0KICAgICAgICAgICAgICAgIGVtYmVkZGluZyA9IHNlbGYucHJvamVjdGlvbihlbWJlZGRpbmcpDQogICAgICAgIA0KICAgICAgICBpZiBlbWJlZGRpbmcuZGltKCkgPT0gMToNCiAgICAgICAgICAgIGVtYmVkZGluZyA9IGVtYmVkZGluZy51bnNxdWVlemUoMCkNCiAgICAgICAgcmV0dXJuIGVtYmVkZGluZy51bnNxdWVlemUoMCkNCg0KDQpjbGFzcyBIeWJyaWRUaXRhblJBRyhubi5Nb2R1bGUpOg0KICAgICIiIg0KICAgIEh5YnJpZCBUaXRhbjogVGl0YW5SQUcgUmV0cmlldmFsICsgTExNIEdlbmVyYXRpb24NCiAgICANCiAgICBUaGlzIGNvbWJpbmVzOg0KICAgIDEuIFRpdGFuUkFHIGZvciBzZW1hbnRpYyBtZW1vcnktYmFzZWQgcmV0cmlldmFsDQogICAgMi4gRXh0ZXJuYWwgTExNIChGbGFuLVQ1LTIpIGZvciBhbnN3ZXIgZ2VuZXJhdGlvbg0KICAgICIiIg0KICAgIA0KICAgIGRlZiBfX2luaXRfXyhzZWxmLCB0aXRhbl9yYWcsIGVtYmVkZGVyLCBsbG1fbmFtZT0iZ29vZ2xlL2ZsYW4tdDUtbGFyZ2UiKToNCiAgICAgICAgc3VwZXIoKS5fX2luaXRfXygpDQogICAgICAgIHNlbGYudGl0YW5fcmFnID0gdGl0YW5fcmFnDQogICAgICAgIHNlbGYuZW1iZWRkZXIgPSBlbWJlZGRlcg0KICAgICAgICANCiAgICAgICAgIyBMb2FkIEZsYW4tVDUgZm9yIGdlbmVyYXRpb24gKGJldHRlciBpbnN0cnVjdGlvbiBmb2xsb3dpbmcgdGhhbiBGbGFuLVQ1LTIpDQogICAgICAgIGZyb20gdHJhbnNmb3JtZXJzIGltcG9ydCBUNUZvckNvbmRpdGlvbmFsR2VuZXJhdGlvbiwgVDVUb2tlbml6ZXINCiAgICAgICAgc2VsZi50b2tlbml6ZXIgPSBUNVRva2VuaXplci5mcm9tX3ByZXRyYWluZWQobGxtX25hbWUpDQogICAgICAgIHNlbGYubGxtID0gVDVGb3JDb25kaXRpb25hbEdlbmVyYXRpb24uZnJvbV9wcmV0cmFpbmVkKGxsbV9uYW1lKQ0KICAgICAgICANCiAgICAgICAgcHJpbnQoZiLinIUgTG9hZGVkIExMTToge2xsbV9uYW1lfSIpDQogICAgDQogICAgZGVmIHJldHJpZXZlKHNlbGYsIHF1ZXN0aW9uLCBlc3NheV9saW5lcywgbGluZV9lbWJlZGRpbmdzLCB0b3BrPTUpOg0KICAgICAgICAiIiINCiAgICAgICAgUmV0cmlldmUgcmVsZXZhbnQgY29udGV4dCB1c2luZyBFbnNlbWJsZSBGdXNpb24uDQogICAgICAgIChDb21iaW5lcyBLZXl3b3JkICsgTWVtb3J5ICsgRW1iZWRkaW5nIGZvciByb2J1c3QgcmV0cmlldmFsKQ0KICAgICAgICANCiAgICAgICAgUmV0dXJuczoNCiAgICAgICAgICAgIGNvbnRleHQ6IFJldHJpZXZlZCB0ZXh0DQogICAgICAgICAgICBkZXRhaWxzOiBEaWN0IHdpdGggcmV0cmlldmFsIHNjb3Jlcw0KICAgICAgICAiIiINCiAgICAgICAgZGltID0gc2VsZi5lbWJlZGRlci50YXJnZXRfZGltDQogICAgICAgIA0KICAgICAgICAjIEVtYmVkIHF1ZXN0aW9uDQogICAgICAgIHF1ZXJ5X2VtYiA9IHNlbGYuZW1iZWRkZXIocXVlc3Rpb24pDQogICAgICAgIHF1ZXJ5X2ZsYXQgPSBxdWVyeV9lbWIucmVzaGFwZSgtMSwgZGltKQ0KICAgICAgICANCiAgICAgICAgd2l0aCB0b3JjaC5ub19ncmFkKCk6DQogICAgICAgICAgICAjIE1ldGhvZCAxOiBLZXl3b3JkIHNjb3Jlcw0KICAgICAgICAgICAgcXVlc3Rpb25fbG93ZXIgPSBxdWVzdGlvbi5sb3dlcigpDQogICAgICAgICAgICBrZXl3b3JkX3Njb3JlcyA9IFtdDQogICAgICAgICAgICBmb3IgbGluZSBpbiBlc3NheV9saW5lczoNCiAgICAgICAgICAgICAgICBzY29yZSA9IHN1bSgxIGZvciB3b3JkIGluIHF1ZXN0aW9uX2xvd2VyLnNwbGl0KCkgDQogICAgICAgICAgICAgICAgICAgICAgICAgICBpZiBsZW4od29yZCkgPiAzIGFuZCB3b3JkIGluIGxpbmUubG93ZXIoKSkNCiAgICAgICAgICAgICAgICBrZXl3b3JkX3Njb3Jlcy5hcHBlbmQoZmxvYXQoc2NvcmUpKQ0KICAgICAgICAgICAga2V5d29yZF9zY29yZXMgPSB0b3JjaC50ZW5zb3Ioa2V5d29yZF9zY29yZXMpDQogICAgICAgICAgICANCiAgICAgICAgICAgICMgTWV0aG9kIDI6IE1lbW9yeSBzY29yZXMgKExUTSBzZW1hbnRpYykNCiAgICAgICAgICAgIG1lbW9yeV9vdXRwdXQgPSBzZWxmLnRpdGFuX3JhZy50aXRhbi5sdG0uZm9yd2FyZF9ub191cGRhdGUocXVlcnlfZmxhdCkNCiAgICAgICAgICAgIG1lbW9yeV92ZWMgPSBtZW1vcnlfb3V0cHV0Lm1lYW4oZGltPTApDQogICAgICAgICAgICBtZW1vcnlfc2NvcmVzID0gdG9yY2gubm4uZnVuY3Rpb25hbC5jb3NpbmVfc2ltaWxhcml0eSgNCiAgICAgICAgICAgICAgICBtZW1vcnlfdmVjLnVuc3F1ZWV6ZSgwKSwgbGluZV9lbWJlZGRpbmdzDQogICAgICAgICAgICApDQogICAgICAgICAgICANCiAgICAgICAgICAgICMgTWV0aG9kIDM6IERpcmVjdCBlbWJlZGRpbmcgc2ltaWxhcml0eQ0KICAgICAgICAgICAgcXVlcnlfdmVjID0gcXVlcnlfZmxhdC5tZWFuKGRpbT0wKQ0KICAgICAgICAgICAgZW1iZWRkaW5nX3Njb3JlcyA9IHRvcmNoLm5uLmZ1bmN0aW9uYWwuY29zaW5lX3NpbWlsYXJpdHkoDQogICAgICAgICAgICAgICAgcXVlcnlfdmVjLnVuc3F1ZWV6ZSgwKSwgbGluZV9lbWJlZGRpbmdzDQogICAgICAgICAgICApDQogICAgICAgICAgICANCiAgICAgICAgICAgICMgTm9ybWFsaXplDQogICAgICAgICAgICBkZWYgbm9ybWFsaXplKHNjb3Jlcyk6DQogICAgICAgICAgICAgICAgbWluX3MsIG1heF9zID0gc2NvcmVzLm1pbigpLCBzY29yZXMubWF4KCkNCiAgICAgICAgICAgICAgICBpZiBtYXhfcyAtIG1pbl9zIDwgMWUtNjoNCiAgICAgICAgICAgICAgICAgICAgcmV0dXJuIHRvcmNoLnplcm9zX2xpa2Uoc2NvcmVzKQ0KICAgICAgICAgICAgICAgIHJldHVybiAoc2NvcmVzIC0gbWluX3MpIC8gKG1heF9zIC0gbWluX3MpDQogICAgICAgICAgICANCiAgICAgICAgICAgICMgRW5zZW1ibGUgRnVzaW9uIChiYWxhbmNlZCB3ZWlnaHRzKQ0KICAgICAgICAgICAgZnVzZWRfc2NvcmVzID0gKA0KICAgICAgICAgICAgICAgIDAuMiAqIG5vcm1hbGl6ZShrZXl3b3JkX3Njb3JlcykgKw0KICAgICAgICAgICAgICAgIDAuNSAqIG5vcm1hbGl6ZShtZW1vcnlfc2NvcmVzKSArDQogICAgICAgICAgICAgICAgMC4zICogbm9ybWFsaXplKGVtYmVkZGluZ19zY29yZXMpDQogICAgICAgICAgICApDQogICAgICAgICAgICANCiAgICAgICAgICAgICMgR2V0IHRvcC1rIGluZGljZXMNCiAgICAgICAgICAgIHRvcGtfdmFsdWVzLCB0b3BrX2luZGljZXMgPSBmdXNlZF9zY29yZXMudG9wayhtaW4odG9waywgbGVuKGVzc2F5X2xpbmVzKSkpDQogICAgICAgICAgICANCiAgICAgICAgICAgICMgQnVpbGQgY29udGV4dCBmcm9tIHRvcC1rIGxpbmVzDQogICAgICAgICAgICBjb250ZXh0X2xpbmVzID0gW2Vzc2F5X2xpbmVzW2lkeF0gZm9yIGlkeCBpbiB0b3BrX2luZGljZXMudG9saXN0KCldDQogICAgICAgICAgICBjb250ZXh0ID0gIiAiLmpvaW4oY29udGV4dF9saW5lcykNCiAgICAgICAgDQogICAgICAgIHJldHVybiBjb250ZXh0LCB7DQogICAgICAgICAgICAia2V5d29yZF90b3AxIjoga2V5d29yZF9zY29yZXMuYXJnbWF4KCkuaXRlbSgpLA0KICAgICAgICAgICAgIm1lbW9yeV90b3AxIjogbWVtb3J5X3Njb3Jlcy5hcmdtYXgoKS5pdGVtKCksDQogICAgICAgICAgICAiZW1iZWRfdG9wMSI6IGVtYmVkZGluZ19zY29yZXMuYXJnbWF4KCkuaXRlbSgpLA0KICAgICAgICAgICAgImZ1c2VkX3RvcDEiOiBmdXNlZF9zY29yZXMuYXJnbWF4KCkuaXRlbSgpLA0KICAgICAgICAgICAgInRvcGtfaW5kaWNlcyI6IHRvcGtfaW5kaWNlcy50b2xpc3QoKQ0KICAgICAgICB9DQogICAgDQogICAgZGVmIGdlbmVyYXRlKHNlbGYsIHF1ZXN0aW9uLCBjb250ZXh0LCBtYXhfbmV3X3Rva2Vucz0xMDApOg0KICAgICAgICAiIiINCiAgICAgICAgR2VuZXJhdGUgYW5zd2VyIHVzaW5nIEZsYW4tVDUgd2l0aCByZXRyaWV2ZWQgY29udGV4dC4NCiAgICAgICAgIiIiDQogICAgICAgICMgQnVpbGQgcHJvbXB0IG9wdGltaXplZCBmb3IgRmxhbi1UNSBpbnN0cnVjdGlvbiBmb3JtYXQNCiAgICAgICAgcHJvbXB0ID0gZiIiIkJhc2VkIG9uIHRoZSBmb2xsb3dpbmcgY29udGV4dCwgYW5zd2VyIHRoZSBxdWVzdGlvbiB3aXRoIHNwZWNpZmljIGZhY3RzIGFuZCBudW1iZXJzLg0KDQpDb250ZXh0OiB7Y29udGV4dH0NCg0KUXVlc3Rpb246IHtxdWVzdGlvbn0NCg0KQW5zd2VyOiIiIg0KICAgICAgICANCiAgICAgICAgIyBUb2tlbml6ZQ0KICAgICAgICBpbnB1dHMgPSBzZWxmLnRva2VuaXplcigNCiAgICAgICAgICAgIHByb21wdCwgDQogICAgICAgICAgICByZXR1cm5fdGVuc29ycz0icHQiLCANCiAgICAgICAgICAgIHRydW5jYXRpb249VHJ1ZSwgDQogICAgICAgICAgICBtYXhfbGVuZ3RoPTUxMg0KICAgICAgICApDQogICAgICAgIA0KICAgICAgICAjIEdlbmVyYXRlIChncmVlZHkgZGVjb2RpbmcgZm9yIG1vcmUgYWNjdXJhdGUgZXh0cmFjdGlvbikNCiAgICAgICAgd2l0aCB0b3JjaC5ub19ncmFkKCk6DQogICAgICAgICAgICBvdXRwdXRzID0gc2VsZi5sbG0uZ2VuZXJhdGUoDQogICAgICAgICAgICAgICAgKippbnB1dHMsDQogICAgICAgICAgICAgICAgbWF4X25ld190b2tlbnM9bWF4X25ld190b2tlbnMsDQogICAgICAgICAgICAgICAgbnVtX2JlYW1zPTMsICAjIEJlYW0gc2VhcmNoIGZvciBiZXR0ZXIgcXVhbGl0eQ0KICAgICAgICAgICAgICAgIGVhcmx5X3N0b3BwaW5nPVRydWUNCiAgICAgICAgICAgICkNCiAgICAgICAgDQogICAgICAgICMgRGVjb2RlDQogICAgICAgIGFuc3dlciA9IHNlbGYudG9rZW5pemVyLmRlY29kZShvdXRwdXRzWzBdLCBza2lwX3NwZWNpYWxfdG9rZW5zPVRydWUpDQogICAgICAgIA0KICAgICAgICByZXR1cm4gYW5zd2VyLCBwcm9tcHQNCiAgICANCiAgICBkZWYgYW5zd2VyKHNlbGYsIHF1ZXN0aW9uLCBlc3NheV9saW5lcywgbGluZV9lbWJlZGRpbmdzLCBleHBlY3RlZF9hbnN3ZXI9Tm9uZSwgdG9waz01LCBtYXhfbmV3X3Rva2Vucz0xMDApOg0KICAgICAgICAiIiINCiAgICAgICAgRnVsbCBwaXBlbGluZTogUmV0cmlldmUgY29udGV4dCwgZ2VuZXJhdGUgYW5zd2VyLCBhbmQgbGVhcm4gZnJvbSBRJkEuDQogICAgICAgIA0KICAgICAgICBBcmdzOg0KICAgICAgICAgICAgcXVlc3Rpb246IFRoZSBxdWVzdGlvbiB0byBhbnN3ZXINCiAgICAgICAgICAgIGVzc2F5X2xpbmVzOiBMaXN0IG9mIGVzc2F5IGxpbmVzDQogICAgICAgICAgICBsaW5lX2VtYmVkZGluZ3M6IFRlbnNvciBvZiBsaW5lIGVtYmVkZGluZ3MNCiAgICAgICAgICAgIGV4cGVjdGVkX2Fuc3dlcjogSWYgcHJvdmlkZWQsIG1vZGVsIGxlYXJucyBmcm9tIHRoaXMgUSZBIHBhaXIgKE9ubGluZSBMZWFybmluZykNCiAgICAgICAgICAgIHRvcGs6IE51bWJlciBvZiBjb250ZXh0IGxpbmVzIHRvIHJldHJpZXZlDQogICAgICAgICAgICBtYXhfbmV3X3Rva2VuczogTWF4IHRva2VucyB0byBnZW5lcmF0ZQ0KICAgICAgICAiIiINCiAgICAgICAgZGltID0gc2VsZi5lbWJlZGRlci50YXJnZXRfZGltDQogICAgICAgIA0KICAgICAgICAjIEVtYmVkIHF1ZXN0aW9uDQogICAgICAgIHF1ZXJ5X2VtYiA9IHNlbGYuZW1iZWRkZXIocXVlc3Rpb24pDQogICAgICAgIHF1ZXJ5X2ZsYXQgPSBxdWVyeV9lbWIucmVzaGFwZSgtMSwgZGltKQ0KICAgICAgICANCiAgICAgICAgIyBTdGVwIDE6IFJldHJpZXZlDQogICAgICAgIGNvbnRleHQsIHJldHJpZXZhbF9kZXRhaWxzID0gc2VsZi5yZXRyaWV2ZSgNCiAgICAgICAgICAgIHF1ZXN0aW9uLCBlc3NheV9saW5lcywgbGluZV9lbWJlZGRpbmdzLCB0b3BrDQogICAgICAgICkNCiAgICAgICAgDQogICAgICAgICMgU3RlcCAyOiBHZW5lcmF0ZQ0KICAgICAgICBhbnN3ZXIsIHByb21wdCA9IHNlbGYuZ2VuZXJhdGUocXVlc3Rpb24sIGNvbnRleHQsIG1heF9uZXdfdG9rZW5zKQ0KICAgICAgICANCiAgICAgICAgIyBTdGVwIDM6IE9ubGluZSBMZWFybmluZyAoVHJ1ZSBUVFQhKSAtIGlmIGV4cGVjdGVkIGFuc3dlciBwcm92aWRlZA0KICAgICAgICBsZWFybmVkID0gRmFsc2UNCiAgICAgICAgaWYgZXhwZWN0ZWRfYW5zd2VyIGlzIG5vdCBOb25lOg0KICAgICAgICAgICAgZXhwZWN0ZWRfZW1iID0gc2VsZi5lbWJlZGRlcihleHBlY3RlZF9hbnN3ZXIpDQogICAgICAgICAgICBleHBlY3RlZF92ZWMgPSBleHBlY3RlZF9lbWIucmVzaGFwZSgtMSwgZGltKS5tZWFuKGRpbT0wKQ0KICAgICAgICAgICAgDQogICAgICAgICAgICAjIFVwZGF0ZSBtZW1vcnk6IFF1ZXN0aW9uIOKGkiBFeHBlY3RlZCBBbnN3ZXINCiAgICAgICAgICAgIHNlbGYudGl0YW5fcmFnLnRpdGFuLmx0bS5mb3J3YXJkX3dpdGhfdXBkYXRlKA0KICAgICAgICAgICAgICAgIHF1ZXJ5X2ZsYXQubWVhbihkaW09MCwga2VlcGRpbT1UcnVlKSwgICMga2V5ID0gcXVlc3Rpb24NCiAgICAgICAgICAgICAgICBleHBlY3RlZF92ZWMudW5zcXVlZXplKDApICAgICAgICAgICAgICAjIHZhbHVlID0gYW5zd2VyDQogICAgICAgICAgICApDQogICAgICAgICAgICBsZWFybmVkID0gVHJ1ZQ0KICAgICAgICANCiAgICAgICAgcmV0dXJuIHsNCiAgICAgICAgICAgICJhbnN3ZXIiOiBhbnN3ZXIsDQogICAgICAgICAgICAiY29udGV4dCI6IGNvbnRleHQsDQogICAgICAgICAgICAicmV0cmlldmFsIjogcmV0cmlldmFsX2RldGFpbHMsDQogICAgICAgICAgICAicHJvbXB0IjogcHJvbXB0LA0KICAgICAgICAgICAgIm9ubGluZV9sZWFybmVkIjogbGVhcm5lZA0KICAgICAgICB9DQoNCg0KZGVmIHJ1bl9oeWJyaWRfdGl0YW5fZGVtbyhhcmdzKToNCiAgICAiIiJSdW4gdGhlIEh5YnJpZCBUaXRhbiBkZW1vLiIiIg0KICAgIA0KICAgIHByaW50KCI9IiAqIDYwKQ0KICAgIHByaW50KCJIWUJSSUQgVElUQU4gREVNTyIpDQogICAgcHJpbnQoIlRpdGFuUkFHIFJldHJpZXZhbCArIEZsYW4tVDUtMiBHZW5lcmF0aW9uIikNCiAgICBwcmludCgiPSIgKiA2MCkNCiAgICANCiAgICAjIFNlbGVjdCBlc3NheSBiYXNlZCBvbiBhcmd1bWVudA0KICAgIGVzc2F5X2tleSA9IGFyZ3MuZXNzYXkubG93ZXIoKQ0KICAgIGlmIGVzc2F5X2tleSA9PSAiY2xpbWF0ZSI6DQogICAgICAgIGVzc2F5X3RleHQgPSBFU1NBWV9DTElNQVRFDQogICAgICAgIGVzc2F5X3F1ZXN0aW9ucyA9IFRFU1RfUVVFU1RJT05TWyJjbGltYXRlIl0NCiAgICBlbGlmIGVzc2F5X2tleSA9PSAiYWkiOg0KICAgICAgICBlc3NheV90ZXh0ID0gRVNTQVlfQUkNCiAgICAgICAgZXNzYXlfcXVlc3Rpb25zID0gVEVTVF9RVUVTVElPTlNbImFpIl0NCiAgICBlbGlmIGVzc2F5X2tleSA9PSAic3BhY2UiOg0KICAgICAgICBlc3NheV90ZXh0ID0gRVNTQVlfU1BBQ0UNCiAgICAgICAgZXNzYXlfcXVlc3Rpb25zID0gVEVTVF9RVUVTVElPTlNbInNwYWNlIl0NCiAgICBlbHNlOg0KICAgICAgICBlc3NheV90ZXh0ID0gRVNTQVlfQ0xJTUFURQ0KICAgICAgICBlc3NheV9xdWVzdGlvbnMgPSBURVNUX1FVRVNUSU9OU1siY2xpbWF0ZSJdDQogICAgDQogICAgcHJpbnQoZiJcbvCfk5YgU2VsZWN0ZWQgRXNzYXk6IHtlc3NheV9rZXkudXBwZXIoKX0iKQ0KICAgIHByaW50KGYiICAgTGVuZ3RoOiB7bGVuKGVzc2F5X3RleHQpfSBjaGFyYWN0ZXJzIikNCiAgICANCiAgICAjIEluaXRpYWxpemUgZW1iZWRkZXINCiAgICBlbWJlZGRlciA9IFNlbnRlbmNlVHJhbnNmb3JtZXJFbWJlZGRlcih0YXJnZXRfZGltPWFyZ3MuZGltKQ0KICAgIA0KICAgICMgSW5pdGlhbGl6ZSBUaXRhblJBRyAoZmlyc3QgY3JlYXRlIGJhc2UgbW9kZWwsIHRoZW4gd3JhcCkNCiAgICBmcm9tIG1haW4gaW1wb3J0IFRpdGFuTUFHDQogICAgDQogICAgYmFzZV9tb2RlbCA9IFRpdGFuTUFHKA0KICAgICAgICBkaW09YXJncy5kaW0sIA0KICAgICAgICB3aW5kb3dfc2l6ZT0zMiwgDQogICAgICAgIGhpZGRlbl9kaW09YXJncy5kaW0gKiAyLA0KICAgICAgICBtZW1vcnlfZGVwdGg9MywgDQogICAgICAgIG51bV9wZXJzaXN0ZW50X3Rva2Vucz00LCANCiAgICAgICAgdGhyZXNob2xkPTAuMA0KICAgICkNCiAgICANCiAgICAjIENvbmZpZ3VyZSBmb3IgaGlnaC1maWRlbGl0eSBtZW1vcnkNCiAgICB3aXRoIHRvcmNoLm5vX2dyYWQoKToNCiAgICAgICAgYmFzZV9tb2RlbC5sdG0udGhldGEuZmlsbF8oMC4wMSkgICAjIExlYXJuaW5nIHJhdGUNCiAgICAgICAgYmFzZV9tb2RlbC5sdG0uYWxwaGEuZmlsbF8oMC4wMDAxKSAjIE1pbmltYWwgZm9yZ2V0dGluZw0KICAgIA0KICAgIHRpdGFuX3JhZyA9IFRpdGFuUkFHKGJhc2VfbW9kZWwpDQogICAgDQogICAgIyBQYXJzZSBlc3NheSBpbnRvIFBBUkFHUkFQSFMgKG5vdCBsaW5lcykgZm9yIGJldHRlciBjb250ZXh0IHJldGVudGlvbg0KICAgICMgU3BsaXQgYnkgZG91YmxlIG5ld2xpbmVzLCB0aGVuIGNsZWFuIHVwIGVhY2ggcGFyYWdyYXBoDQogICAgcmF3X3BhcmFncmFwaHMgPSBlc3NheV90ZXh0LnNwbGl0KCdcblxuJykNCiAgICBlc3NheV9wYXJhZ3JhcGhzID0gW10NCiAgICBmb3IgcGFyYSBpbiByYXdfcGFyYWdyYXBoczoNCiAgICAgICAgIyBNZXJnZSBsaW5lcyB3aXRoaW4gcGFyYWdyYXBoLCBjbGVhbiB1cA0KICAgICAgICBjbGVhbmVkID0gJyAnLmpvaW4obGluZS5zdHJpcCgpIGZvciBsaW5lIGluIHBhcmEuc3BsaXQoJ1xuJykgaWYgbGluZS5zdHJpcCgpKQ0KICAgICAgICBpZiBjbGVhbmVkIGFuZCBsZW4oY2xlYW5lZCkgPiAxMDogICMgU2tpcCBlbXB0eSBvciB2ZXJ5IHNob3J0DQogICAgICAgICAgICBlc3NheV9wYXJhZ3JhcGhzLmFwcGVuZChjbGVhbmVkKQ0KICAgIA0KICAgIHByaW50KGYiXG7wn5SEIEVtYmVkZGluZyB7bGVuKGVzc2F5X3BhcmFncmFwaHMpfSBwYXJhZ3JhcGhzLi4uIikNCiAgICANCiAgICAjIEVtYmVkIGFsbCBwYXJhZ3JhcGhzDQogICAgd2l0aCB0b3JjaC5ub19ncmFkKCk6DQogICAgICAgIHBhcmFfZW1iZWRkaW5ncyA9IFtdDQogICAgICAgIGZvciBwYXJhIGluIGVzc2F5X3BhcmFncmFwaHM6DQogICAgICAgICAgICBwYXJhX2VtYiA9IGVtYmVkZGVyKHBhcmEpDQogICAgICAgICAgICBwYXJhX3ZlYyA9IHBhcmFfZW1iLnJlc2hhcGUoLTEsIGFyZ3MuZGltKS5tZWFuKGRpbT0wKQ0KICAgICAgICAgICAgcGFyYV9lbWJlZGRpbmdzLmFwcGVuZChwYXJhX3ZlYykNCiAgICAgICAgbGluZV9lbWJlZGRpbmdzID0gdG9yY2guc3RhY2socGFyYV9lbWJlZGRpbmdzKSAgIyBLZWVwIHZhcmlhYmxlIG5hbWUgZm9yIGNvbXBhdGliaWxpdHkNCiAgICANCiAgICAjIFVzZSBwYXJhZ3JhcGhzIGluc3RlYWQgb2YgbGluZXMNCiAgICBlc3NheV9saW5lcyA9IGVzc2F5X3BhcmFncmFwaHMgICMgQWxpYXMgZm9yIGNvbXBhdGliaWxpdHkNCiAgICANCiAgICAjIERpZ2VzdCBpbnRvIFRpdGFuUkFHDQogICAgcHJpbnQoZiJcbvCfp6AgRGlnZXN0aW5nIGludG8gVGl0YW5SQUcgKHthcmdzLmVwb2Noc30gZXBvY2hzKS4uLiIpDQogICAgYWxsX2VtYmVkZGluZ3MgPSB0b3JjaC5jYXQoW2VtYmVkZGVyKGxpbmUpIGZvciBsaW5lIGluIGVzc2F5X2xpbmVzXSwgZGltPTEpDQogICAgDQogICAgZm9yIGVwb2NoIGluIHJhbmdlKGFyZ3MuZXBvY2hzKToNCiAgICAgICAgdGl0YW5fcmFnLmRpZ2VzdF9rbm93bGVkZ2UoYWxsX2VtYmVkZGluZ3MpDQogICAgICAgIGlmIChlcG9jaCArIDEpICUgMTAgPT0gMDoNCiAgICAgICAgICAgIHByaW50KGYiICAgRXBvY2gge2Vwb2NoICsgMX0ve2FyZ3MuZXBvY2hzfSBjb21wbGV0ZWQiKQ0KICAgIA0KICAgICMgSW5pdGlhbGl6ZSBIeWJyaWQgVGl0YW4gd2l0aCBGbGFuLVQ1DQogICAgcHJpbnQoIlxu8J+kliBJbml0aWFsaXppbmcgSHlicmlkIFRpdGFuIChGbGFuLVQ1IGdlbmVyYXRvcikuLi4iKQ0KICAgIGh5YnJpZF90aXRhbiA9IEh5YnJpZFRpdGFuUkFHKHRpdGFuX3JhZywgZW1iZWRkZXIpICAjIFVzZXMgZGVmYXVsdCBGbGFuLVQ1DQogICAgDQogICAgIyBSdW4gUSZBIHdpdGggT25saW5lIExlYXJuaW5nDQogICAgcHJpbnQoIlxuIiArICI9IiAqIDYwKQ0KICAgIHByaW50KCJRVUVTVElPTiBBTlNXRVJJTkcgd2l0aCBPTkxJTkUgTEVBUk5JTkciKQ0KICAgIHByaW50KCI9IiAqIDYwKQ0KICAgIA0KICAgIGZvciBpLCAocXVlc3Rpb24sIGV4cGVjdGVkKSBpbiBlbnVtZXJhdGUoZXNzYXlfcXVlc3Rpb25zLCAxKToNCiAgICAgICAgcHJpbnQoZiJcbnsn4pSAJyAqIDYwfSIpDQogICAgICAgIHByaW50KGYiUXtpfToge3F1ZXN0aW9ufSIpDQogICAgICAgIHByaW50KGYiRXhwZWN0ZWQ6IHtleHBlY3RlZH0iKQ0KICAgICAgICANCiAgICAgICAgIyBHZXQgYW5zd2VyIGZyb20gSHlicmlkIFRpdGFuICh3aXRoIE9ubGluZSBMZWFybmluZyEpDQogICAgICAgIHJlc3VsdCA9IGh5YnJpZF90aXRhbi5hbnN3ZXIoDQogICAgICAgICAgICBxdWVzdGlvbiwgDQogICAgICAgICAgICBlc3NheV9saW5lcywgDQogICAgICAgICAgICBsaW5lX2VtYmVkZGluZ3MsDQogICAgICAgICAgICBleHBlY3RlZF9hbnN3ZXI9ZXhwZWN0ZWQsICAjIEVuYWJsZSBPbmxpbmUgTGVhcm5pbmcNCiAgICAgICAgICAgIHRvcGs9YXJncy50b3BrLA0KICAgICAgICAgICAgbWF4X25ld190b2tlbnM9YXJncy5tYXhfdG9rZW5zDQogICAgICAgICkNCiAgICAgICAgDQogICAgICAgIHByaW50KGYiXG5bUmV0cmlldmFsXSBJbmRpY2VzOiB7cmVzdWx0WydyZXRyaWV2YWwnXVsndG9wa19pbmRpY2VzJ119IikNCiAgICAgICAgcHJpbnQoZiJbQ29udGV4dF1cbiAge3Jlc3VsdFsnY29udGV4dCddfSIpDQogICAgICAgIHByaW50KGYiXG5bR2VuZXJhdGVkIEFuc3dlcl0iKQ0KICAgICAgICBwcmludChmIiAge3Jlc3VsdFsnYW5zd2VyJ119IikNCiAgICAgICAgDQogICAgICAgICMgU2hvdyBPbmxpbmUgTGVhcm5pbmcgc3RhdHVzDQogICAgICAgIGlmIHJlc3VsdC5nZXQoJ29ubGluZV9sZWFybmVkJyk6DQogICAgICAgICAgICBwcmludCgiICDwn5OaIE9ubGluZSBMZWFybmluZzogVXBkYXRlZCBtZW1vcnkgd2l0aCB0aGlzIFEmQSIpDQogICAgICAgIA0KICAgICAgICAjIFNpbXBsZSBldmFsdWF0aW9uOiBjaGVjayBpZiBleHBlY3RlZCBhbnN3ZXIga2V5d29yZHMgYXBwZWFyDQogICAgICAgIGV4cGVjdGVkX3dvcmRzID0gZXhwZWN0ZWQubG93ZXIoKS5zcGxpdCgpDQogICAgICAgIGFuc3dlcl9sb3dlciA9IHJlc3VsdFsnYW5zd2VyJ10ubG93ZXIoKQ0KICAgICAgICBtYXRjaGVzID0gc3VtKDEgZm9yIHdvcmQgaW4gZXhwZWN0ZWRfd29yZHMgaWYgbGVuKHdvcmQpID4gMyBhbmQgd29yZCBpbiBhbnN3ZXJfbG93ZXIpDQogICAgICAgIG1hdGNoX3JhdGlvID0gbWF0Y2hlcyAvIGxlbihbdyBmb3IgdyBpbiBleHBlY3RlZF93b3JkcyBpZiBsZW4odykgPiAzXSkgaWYgZXhwZWN0ZWRfd29yZHMgZWxzZSAwDQogICAgICAgIA0KICAgICAgICBpZiBtYXRjaF9yYXRpbyA+IDAuMzoNCiAgICAgICAgICAgIHByaW50KGYiICDinIUgQ29udGFpbnMgZXhwZWN0ZWQga2V5d29yZHMgKHttYXRjaF9yYXRpbzouMCV9KSIpDQogICAgICAgIGVsc2U6DQogICAgICAgICAgICBwcmludChmIiAg4pqg77iPIE1pc3NpbmcgZXhwZWN0ZWQga2V5d29yZHMgKHttYXRjaF9yYXRpbzouMCV9KSIpDQogICAgDQogICAgcHJpbnQoIlxuIiArICI9IiAqIDYwKQ0KICAgIHByaW50KCLinIUgSHlicmlkIFRpdGFuIERlbW8gQ29tcGxldGUhIikNCiAgICBwcmludCgiPSIgKiA2MCkNCiAgICANCiAgICByZXR1cm4gaHlicmlkX3RpdGFuDQoNCg0KZGVmIG1haW4oKToNCiAgICBwYXJzZXIgPSBhcmdwYXJzZS5Bcmd1bWVudFBhcnNlcihkZXNjcmlwdGlvbj0iSHlicmlkIFRpdGFuIERlbW8iKQ0KICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tZXNzYXkiLCB0eXBlPXN0ciwgZGVmYXVsdD0iY2xpbWF0ZSIsDQogICAgICAgICAgICAgICAgICAgICAgIGNob2ljZXM9WyJjbGltYXRlIiwgImFpIiwgInNwYWNlIl0sDQogICAgICAgICAgICAgICAgICAgICAgIGhlbHA9IkVzc2F5IHRvIHVzZSIpDQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1kaW0iLCB0eXBlPWludCwgZGVmYXVsdD0yNTYsDQogICAgICAgICAgICAgICAgICAgICAgIGhlbHA9IkVtYmVkZGluZyBkaW1lbnNpb24iKQ0KICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tZXBvY2hzIiwgdHlwZT1pbnQsIGRlZmF1bHQ9NTAsDQogICAgICAgICAgICAgICAgICAgICAgIGhlbHA9IkRpZ2VzdGlvbiBlcG9jaHMiKQ0KICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tdG9wayIsIHR5cGU9aW50LCBkZWZhdWx0PTMsDQogICAgICAgICAgICAgICAgICAgICAgIGhlbHA9IlRvcC1rIGxpbmVzIHRvIHJldHJpZXZlIikNCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLW1heF90b2tlbnMiLCB0eXBlPWludCwgZGVmYXVsdD01MCwNCiAgICAgICAgICAgICAgICAgICAgICAgaGVscD0iTWF4IHRva2VucyB0byBnZW5lcmF0ZSIpDQogICAgDQogICAgYXJncyA9IHBhcnNlci5wYXJzZV9hcmdzKCkNCiAgICBydW5faHlicmlkX3RpdGFuX2RlbW8oYXJncykNCg0KDQppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOg0KICAgIG1haW4oKQ0K'
}

for path, content in files.items():
    write_file(path, content)

print('Files in current directory:', os.listdir('.'))
if os.path.exists('experiments'):
    print('Files in experiments:', os.listdir('experiments'))


## 3. Run Baseline (Dense Update)

In [ ]:
# Run Baseline (Dense Update)
# Decreasing vocab_size to 16 to ensure model can learn quickly
import os
if os.path.exists('projects/original_benchmarks/train_toy.py'):
    print('Running Baseline (Dense Update, Vocab=16)...')
    !python projects/original_benchmarks/train_toy.py --model_variant MAC --num_epochs 5 --batch_size 16 --vocab_size 16 --threshold 0.0
else:
    print('ERROR: projects/original_benchmarks/train_toy.py not found!')


## 4. Run Innovation (Sparse Update)

In [ ]:
# Run Innovation (Sparse Update)
print('Running Sparse Update (Threshold = 0.5, Vocab=16)...')
!python projects/original_benchmarks/train_toy.py --model_variant MAC --num_epochs 5 --batch_size 16 --vocab_size 16 --threshold 0.5


## 5. TitanRAG Prototype Demo

In [ ]:
# TitanRAG Prototype Demo
# Demonstrating 'Retrieve-then-Digest' mechanism
import torch
import sys, os
sys.path.append(os.path.abspath('src'))
from main import TitanMAC, TitanRAG

print('Initializing TitanRAG...')
# 1. Initialize Base Model
base_model = TitanMAC(dim=128, chunk_size=32, hidden_dim=128, memory_depth=2, num_persistent_tokens=4, threshold=0.1)
rag_model = TitanRAG(base_model)

# 2. Simulate Data (Embeddings)
print('Simulating Retrieval...')
doc1 = torch.randn(1, 64, 128) # Retrieved Doc A
doc2 = torch.randn(1, 64, 128) # Retrieved Doc B
query = torch.randn(1, 16, 128) # User Query

# 3. Run Pipeline
print('Executing Digestion & Inference...')
output = rag_model.query_with_context(query, [doc1, doc2])

print('Success! Output shape:', output.shape)
print('The model weights have been updated on-the-fly with document info.')


## 6. RAG Comparison Experiment (Small Scale)

In [ ]:
# Experiment: RAG Comparison - Small Scale (Within Context Window)
# Both methods should succeed.
print('--- Running RAG Comparison (Small Scale: 10 Docs) ---')
!python projects/hybrid_titans/rag_compare.py --num_docs 10 --doc_len 32


## 7. RAG Comparison Experiment (Large Scale)

In [ ]:
# Experiment: RAG Comparison - Large Scale (Exceeds Context Window)
# Traditional RAG should FAIL (Context Limit).
# TitanRAG should PASS (Infinite Memory).
print('\n--- Running RAG Comparison (Large Scale: 100 Docs) ---')
!python projects/hybrid_titans/rag_compare.py --num_docs 100 --doc_len 32


## 8. Essay-based TitanRAG Demo
This cell demonstrates TitanRAG on real text: Climate Essay.
The model digests the essay and then answers verification questions.

In [ ]:
# Run Essay RAG Demo (Climate Essay - Random Embedder Baseline)
print('--- Essay RAG Demo: Climate (Random Embedder) ---')
!python projects/hybrid_titans/essay_rag_demo.py --essay climate --epochs 50 --embedder random


## 9. Semantic Embedder (sentence-transformers)
Now running with **semantic embeddings** for meaningful text similarity.
This should show significant improvement in Hybrid RAG performance!

In [ ]:
# Run ALL Essays with Semantic Embedder
print('=' * 60)
print('SEMANTIC EMBEDDER TEST - ALL ESSAYS')
print('=' * 60)

# Climate Essay
print('\n--- Essay 1/3: Climate Change ---')
!python projects/hybrid_titans/essay_rag_demo.py --essay climate --epochs 50 --embedder semantic

# AI Essay
print('\n--- Essay 2/3: Artificial Intelligence ---')
!python projects/hybrid_titans/essay_rag_demo.py --essay ai --epochs 50 --embedder semantic

# Space Essay
print('\n--- Essay 3/3: Solar System ---')
!python projects/hybrid_titans/essay_rag_demo.py --essay space --epochs 50 --embedder semantic


## 10. Hybrid Titan: TitanRAG + Flan-T5 Generation
This cell demonstrates **Direction A** from the research plan:
- TitanRAG performs retrieval using Ensemble Fusion (Keyword + Memory + Embedding)
- Flan-T5 generates natural language answers from context
- **Online Learning**: Model learns from each Q&A pair

In [ ]:
# Hybrid Titan Demo: TitanRAG Retrieval + Flan-T5 Generation
# Running ALL THREE ESSAYS with Online Learning

print('=' * 60)
print('HYBRID TITAN DEMO - ALL ESSAYS')
print('=' * 60)

# 1. Climate Essay
print('\n' + '─' * 60)
print('📖 Essay 1/3: CLIMATE CHANGE')
print('─' * 60)
!python projects/hybrid_titans/hybrid_titan_demo.py --essay climate --epochs 50 --topk 5

# 2. AI Essay
print('\n' + '─' * 60)
print('📖 Essay 2/3: ARTIFICIAL INTELLIGENCE')
print('─' * 60)
!python projects/hybrid_titans/hybrid_titan_demo.py --essay ai --epochs 50 --topk 5

# 3. Space Essay
print('\n' + '─' * 60)
print('📖 Essay 3/3: SOLAR SYSTEM')
print('─' * 60)
!python projects/hybrid_titans/hybrid_titan_demo.py --essay space --epochs 50 --topk 5
